# 13 — Build the training data (teacher stage)

**Step 2 of 3.** `12_kaggle_artifact_import` -> **`13_distill_dataset_kaggle`** -> `14_distill_train_eval_kaggle`

The teacher model (12B) reads retrieved context and writes an answer. Those answers become the
training targets for the student model (4B). Nothing is trained here.

## What changed from v1, and why

**v1 had two legs. Now there is one.** The old design mixed 900 real KCC expert answers with
250 teacher answers. The real answers turned out to be the wrong target:

* KCC experts wrote their answers **without seeing any retrieved context**. Our prompt tells
  the model "answer only from the context" and "end with `Sources: [n]`". The expert answers do
  neither. Training on them teaches the model to ignore the context and skip citations.
* Those answers contain dose numbers that are not in the retrieved chunks. Training on them
  teaches the model to state numbers it cannot support — the exact risk this project is trying
  to reduce.

So every training answer now comes from the teacher, which sees the same rules and the same
context the student will see. The KCC expert answer is still saved next to each row as
`kcc_reference_answer`, but only for humans to read. It is never a training target.

This also makes the method easier to describe: it is plain sequence-level knowledge
distillation. The teacher's text is the target. There is no KL term, because storing full
teacher logits for a 262k vocabulary is not possible at this scale.

## The language problem, and the fix

The KCC corpus has a strange split: **questions are 99.98% English, answers are 98.8% Hindi.**

That breaks the old design in two ways. The script-agreement filter (keep rows where question
and answer use the same script) throws away almost everything, and what survives is nearly all
English-to-English. So the student would only ever see English.

Three fixes, all in this notebook:

1. **Language detection now has three outcomes**, not two: `hi` (Devanagari), `hinglish`
   (Hindi words in Roman letters), and `en`. The old version could not see Hinglish at all,
   because Hinglish uses the same letters as English.
2. **The prompt states the answer language at the very end**, right before the model starts
   writing, and tells it not to copy the language of the context. Retrieved chunks are mostly
   Hindi, so without this the model copies them.
3. **Some English questions are rewritten into Hindi and Hinglish** before retrieval, so the
   training set is not English-only. The corpus cannot supply these, so we make them.

## Three fixes from the first real run

Looking at what the teacher actually wrote the first time showed three problems, all fixed here:

1. **It described where information is kept instead of giving advice.** Answers like
   "information about bakani disease is available for Budaun, Shamli and Aligarh" were common.
   The cause was the source label above each context item: the model read `crop=..., district=...,
   year=...` as content and summarised it. Districts and years appear nowhere else, which is how
   we know. Now the label is clearly marked `SOURCE (for citation only, do not repeat)` and the
   chunk text is marked `CONTENT`, with a rule saying never to mention districts, years or file
   names.
2. **Citations appeared inside sentences.** `Use 30 kg urea [1] and then irrigate` instead of a
   clean paragraph followed by one `Sources:` line. There is now an explicit rule and a check
   that drops any answer with a bracket number before the last line.
3. **English questions got Hindi answers**, because the retrieved chunks are mostly Hindi and
   the model copied them. Fixed by naming the answer language last, right before the model
   writes.

## What the smoke run showed, and what changed

Half the teacher's answers were clean advice. The other half shared one fault: when the
retrieved chunks did not answer the question, the model **described what the chunks were
about** instead of simply refusing.

    Ask about fenugreek. Ask about strawberry. Ask about sugarcane.

    I do not have information about plant protection for Brinjal.
    I can provide information about cultivation and planting times.

The second one is the tricky case. It refuses correctly, then spoils it by offering other
topics. The first version of the filter let it through, because it saw the refusal and
stopped checking. Training on it would teach the student to answer a maize question by
offering mango information.

Two fixes: rule 3 now says the refusal is the *whole* answer, with worked examples of the
bad forms, and `offers_alternatives()` runs **before** the refusal check instead of after.

Question rewriting is also **off** now -- see the note in step 0.

## Teaching the model to say "I do not have that"

Refusal is currently handled only by the search score: anything scoring below the threshold
never reaches the model. So the model has never practised refusing, and if that gate is ever
bypassed it will answer anything.

This notebook now adds two kinds of training row for it:

* **wrong context** — a real farming question shown someone else's retrieved text. The right
  answer is "the context does not cover this".
* **off topic** — a question with nothing to do with farming, and no context at all.

The teacher writes both, so the wording varies instead of the model learning one fixed
sentence. Rows where the teacher answered anyway are thrown out, because they would teach
exactly the wrong habit.

## What you need

| Need | Where |
|---|---|
| Notebook 12's output attached | Add Input -> Your Work -> Notebook Output |
| `HF_TOKEN` secret, attached to this notebook | Add-ons -> Secrets |
| GPU T4 x2, Internet **On** | Settings |

Set `SMOKE_TEST = True` for a short test run first. Only turn it off once that works.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Retrieve the token from Kaggle Secrets
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("AIPIPE_TOKEN")   # Use the exact name you gave

# Optionally set it as an environment variable if other code expects it
os.environ["AIPIPE_TOKEN"] = token

In [2]:
import subprocess, sys, importlib.metadata as _md
from packaging.version import Version

def _ver(pkg):
    try:
        return Version(_md.version(pkg))
    except Exception:
        return None

# Kaggle already ships torch, numpy and transformers, built against each other. If pip is
# allowed to move numpy, torch stops importing with a confusing message about METH_CLASS.
# A "Save & Run All" job has no kernel to restart, so we pin instead of recovering.
_PIN = ("numpy", "torch", "torchvision", "torchaudio", "scipy", "pandas", "pyarrow")
_lines = [f"{p}=={_md.version(p)}" for p in _PIN if _ver(p) is not None]
_CONSTRAINTS = "/tmp/pip-constraints.txt"
with open(_CONSTRAINTS, "w") as f:
    f.write("\n".join(_lines) + "\n")
print("pinned (pip may not change these):")
for _l in _lines:
    print(f"   {_l}")

_NEED = [("qdrant-client", None), ("sentence-transformers", "3.0.0"),
         ("transformers", "4.50.0"), ("accelerate", "0.30.0"),
         ("bitsandbytes", "0.43.0"), ("sentencepiece", None), ("hf_transfer", None)]
_missing = []
for _pkg, _minv in _NEED:
    _v = _ver(_pkg)
    if _v is None:
        _missing.append(_pkg)
    elif _minv and _v < Version(_minv):
        _missing.append(f"{_pkg}>={_minv}")

if _missing:
    print(f"\ninstalling: {_missing}")
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                         "--no-warn-conflicts", "-c", _CONSTRAINTS, *_missing])
    if _r.returncode != 0:
        print("[WARN] pip returned an error -- the import check below is what matters")
else:
    print("\nall packages already present, nothing installed")

try:
    import torch
    print(f"\ntorch {torch.__version__}   cuda={torch.cuda.is_available()}   "
          f"gpus={torch.cuda.device_count()}")
except Exception as _e:
    raise RuntimeError(
        f"torch is broken in this session: {type(_e).__name__}: {_e}\n\n"
        "  1. Right panel -> Session options -> STOP session, then open the notebook again.\n"
        "     A restart keeps the same container, so a numpy that pip replaced stays\n"
        "     replaced. Stopping gives you the original image back.\n"
        "  2. Run All again.\n") from _e

pinned (pip may not change these):
   numpy==2.0.2
   torch==2.10.0+cu128
   torchvision==0.25.0+cu128
   torchaudio==2.10.0+cu128
   scipy==1.16.3
   pandas==2.3.3
   pyarrow==24.0.0

installing: ['qdrant-client', 'bitsandbytes', 'hf_transfer']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.9 MB/s eta 0:00:00

torch 2.10.0+cu128   cuda=True   gpus=2


## 0. Settings

`SMOKE_TEST = True` uses a small sample so you can check the whole notebook runs. Set it to
`False` for the real run.

In [3]:
import os, sys, json, re, time, random, hashlib, gc, shutil, glob, math
from collections import Counter

os.environ.setdefault("HF_HOME", "/kaggle/temp/hf_cache" if os.path.isdir("/kaggle") else "./hf_cache")
# Set before any GPU memory is used. Loading weights makes many small allocations, and on a
# nearly full card the failure is fragmentation, not real exhaustion.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
try:
    import hf_transfer  # noqa: F401
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except ImportError:
    pass

import torch
import pandas as pd

SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)

# =========================== MAIN SWITCH ===============================================
SMOKE_TEST = False
# =======================================================================================

IS_KAGGLE = os.path.isdir("/kaggle/input") or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
WORK_ROOT = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else os.path.abspath("./work")
OUT_ROOT  = "/kaggle/working" if IS_KAGGLE else os.path.abspath("./out")
OUTPUT_DIR = os.path.join(OUT_ROOT, "distill_data")
os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- how many training rows -----------------------------------------------------------
# Sized for the filters, not for the final count. Measured on the first real run, roughly
# half the teacher's answers get dropped (numbers not in context, describing where info
# lives, citations mid-sentence, no Sources line). So ask for about twice what you want.
N_QUESTIONS   = 24 if SMOKE_TEST else 2500   # questions the teacher will answer
RETRIEVAL_TOP_K = 5
CTX_TOP_K     = 5
CTX_CHAR_CAP  = 700

# ---- language mix ----------------------------------------------------------------------
# KCC questions are 99.98% English. If we train only on those, the student never sees a Hindi
# or Hinglish question. We rewrite some of them so it does. 0.0 turns this off.
# OFF by default. The 4B model was asked to rewrite the questions and got crop names
# wrong: "insect control in MINT" came back as a question about matar (peas), and "cumin"
# came back as kheera (cucumber). The crop is the one word that has to survive, because
# retrieval keys on it -- a wrong crop fetches the wrong chunks and the teacher then
# answers the wrong question perfectly. Wrong training rows are worse than English-only
# ones, since the prompt names the answer language at serving time anyway.
#
# Turn this back on only with a stronger translator, and read the printed samples before
# trusting them.
MAKE_OTHER_LANGUAGES = True
HINDI_FRACTION    = 0.20     # share of questions rewritten into Hindi
HINGLISH_FRACTION = 0.20     # share rewritten into Hinglish

# ---- teaching the model to refuse -------------------------------------------------------
# Without these rows the student never learns to say "I do not have that". Refusal is
# currently done entirely by the search score gate: anything below the threshold never
# reaches the model, so the model itself has never practised it. If that gate is ever
# bypassed, or a question scores just above it while the context is useless, the model
# answers anyway.
#
# Two kinds of row fix that:
#   "no_context"    -- a real farming question paired with someone else's context. The right
#                      answer is "the context does not cover this", not a guess.
#   "off_topic"     -- a question with nothing to do with farming and no context at all.
# The teacher writes both, so the wording varies instead of the model memorising one string.
ADD_REFUSAL_ROWS = True
NO_CONTEXT_FRACTION = 0.25                     # was 0.10
OFF_TOPIC_COUNT     = 20 if SMOKE_TEST else 250   # was 6 / 60

# ---- teacher ---------------------------------------------------------------------------
# Same weights as google/gemma-3-12b-it, but already stored in 4-bit. Loading the official
# repo with a BitsAndBytesConfig fails on transformers 5.x: the quantization setting reaches
# the device-map planner but not the code that writes the weights, so a plan made for 7 GB
# gets 24 GB of fp16 written into it and runs out of memory half way through.
# No fallback to a different model: the teacher's name is part of the method.
TEACHER_MODEL_ID   = "unsloth/gemma-3-12b-it-bnb-4bit"
TEACHER_4BIT       = True
TEACHER_BATCH_SIZE = 4 if SMOKE_TEST else 8   # halves itself if it runs out of memory
TEACHER_TEMPERATURE = 0.4
ANSWER_MAX_TOKENS  = 160    # 100 cuts the "Sources:" line off the end of the answer
REQUIRE_CITATION   = True   # drop teacher answers with no Sources line at all
PURGE_TEACHER_CACHE = True  # frees ~8 GB before notebook 14 downloads the student

# The student is not trained here. We load it only to (a) build prompts with its chat
# template and (b) rewrite questions into other languages.
STUDENT_MODEL_ID = "google/gemma-3-4b-it"
STUDENT_LOAD_IDS = ["unsloth/gemma-3-4b-it-bnb-4bit", "google/gemma-3-4b-it"]

QDRANT_URL = "http://localhost:6333"

if "HF_TOKEN" not in os.environ:
    for loader in (
        lambda: __import__("kaggle_secrets").UserSecretsClient().get_secret("HF_TOKEN"),
        lambda: __import__("google.colab", fromlist=["userdata"]).userdata.get("HF_TOKEN"),
    ):
        try:
            _t = loader()
            if _t:
                os.environ["HF_TOKEN"] = _t
                print("HF_TOKEN loaded from secrets")
                break
        except Exception:
            pass
if "HF_TOKEN" not in os.environ:
    print("[WARN] no HF_TOKEN. Gemma needs one -- step 9 will fail with a 401.")


# ---------------------------------------------------------------------------------------
# Progress logging. Every big step prints a banner with its number, name and expected time,
# and the summary at the end shows what each one actually took.
# ---------------------------------------------------------------------------------------
RUN_START = time.time()
STEP_LOG = []
_step_state = {"n": None, "name": None, "t0": None}

PLAN = [
    (1,  "Find the input files",            0.2,  0.2),
    (2,  "Start the Qdrant server",         1.0,  1.0),
    (3,  "Load the search index",           6.0,  9.0),
    (4,  "Load the embedding model",        1.5,  2.0),
    (5,  "Set up name cleanup",             0.1,  0.1),
    (6,  "Set up the search function",      0.2,  0.2),
    (7,  "Set up prompts and scorers",      0.7,  0.7),
    (8,  "Load the KCC questions",          0.3,  0.8),
    (9,  "Rewrite questions into hi/hing",  0.1,  0.1),
    (10, "Search context + refusal rows",   0.5,  6.0),
    (11, "Search context for eval Qs",      0.3,  0.3),
    (12, "Shut down search, save progress", 0.5,  0.5),
    (13, "Teacher writes the answers",      6.0, 50.0),
    (14, "Save the output files",           0.2,  0.5),
]
PLAN_MIN = {n: (s if SMOKE_TEST else f) for n, _, s, f in PLAN}


def step(n, name):
    """Start a step. Closes the previous one and records how long it took."""
    if _step_state["n"] is not None:
        took = (time.time() - _step_state["t0"]) / 60
        STEP_LOG.append((_step_state["n"], _step_state["name"], round(took, 2)))
        print(f"\n--- step {_step_state['n']} done in {took:.1f} min "
              f"(total so far {(time.time()-RUN_START)/60:.1f} min) ---")
    _step_state.update(n=n, name=name, t0=time.time())
    done = sum(PLAN_MIN[k] for k in PLAN_MIN if k < n)
    left = sum(PLAN_MIN[k] for k in PLAN_MIN if k >= n)
    print("\n" + "=" * 78)
    print(f"STEP {n}/{len(PLAN)}  {name}")
    print(f"expected ~{PLAN_MIN.get(n, 0):.1f} min   |   about {left:.0f} min left of "
          f"~{done+left:.0f} min total")
    print("=" * 78)


def finish_steps():
    if _step_state["n"] is not None:
        took = (time.time() - _step_state["t0"]) / 60
        STEP_LOG.append((_step_state["n"], _step_state["name"], round(took, 2)))
        _step_state["n"] = None


def free_vram(*names):
    """Delete the named globals and give the memory back."""
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def disk_free_gb(path=WORK_ROOT):
    st = os.statvfs(path)
    return st.f_bavail * st.f_frsize / 1e9


def vram_report(tag=""):
    if not torch.cuda.is_available():
        print(f"[{tag}] no GPU")
        return
    parts = [f"gpu{i} {torch.cuda.mem_get_info(i)[0]/1e9:.1f} GB free"
             for i in range(torch.cuda.device_count())]
    print(f"[{tag}] " + "   ".join(parts) + f"   disk {disk_free_gb():.1f} GB free")


print(f"mode       : {'SMOKE TEST (small sample)' if SMOKE_TEST else 'FULL RUN'}")
print(f"questions  : {N_QUESTIONS}")
print(f"teacher    : {TEACHER_MODEL_ID}")
print(f"output     : {OUTPUT_DIR}")
print(f"\nexpected total time: about {sum(PLAN_MIN.values()):.0f} minutes")
print("\nplan:")
for n, name, s, f in PLAN:
    print(f"   {n:>2}. {name:<34} ~{(s if SMOKE_TEST else f):>5.1f} min")
if SMOKE_TEST:
    print("\n" + "!" * 78)
    print("!! SMOKE_TEST = True. Small sample. Set it to False for the real run.")
    print("!" * 78)
vram_report("start")

HF_TOKEN loaded from secrets
mode       : FULL RUN
questions  : 2500
teacher    : unsloth/gemma-3-12b-it-bnb-4bit
output     : /kaggle/working/distill_data

expected total time: about 71 minutes

plan:
    1. Find the input files               ~  0.2 min
    2. Start the Qdrant server            ~  1.0 min
    3. Load the search index              ~  9.0 min
    4. Load the embedding model           ~  2.0 min
    5. Set up name cleanup                ~  0.1 min
    6. Set up the search function         ~  0.2 min
    7. Set up prompts and scorers         ~  0.7 min
    8. Load the KCC questions             ~  0.8 min
    9. Rewrite questions into hi/hing     ~  0.1 min
   10. Search context + refusal rows      ~  6.0 min
   11. Search context for eval Qs         ~  0.3 min
   12. Shut down search, save progress    ~  0.5 min
   13. Teacher writes the answers         ~ 50.0 min
   14. Save the output files              ~  0.5 min
[start] gpu0 15.5 GB free   gpu1 15.5 GB free   disk 20.

## 1. Find the input files

`manifest.json` tells us where everything is. It also holds the settings the search index was
built with: model name, vector size, and the score thresholds. We read them instead of typing
them again, because a wrong threshold silently refuses every question.

In [4]:
step(1, "Find the input files")

def _glob1(pat):
    hits = sorted(glob.glob(pat, recursive=True))
    return hits[0] if hits else None

SEARCH_ROOTS = ["/kaggle/input", OUT_ROOT, "."]

_man = None
for root in SEARCH_ROOTS:
    _man = _glob1(os.path.join(root, "**", "manifest.json"))
    if _man:
        break
if not _man:
    print("what is in /kaggle/input:")
    for r, d, fs in os.walk("/kaggle/input"):
        if r.replace("/kaggle/input", "").count(os.sep) > 4:
            d[:] = []
            continue
        for f in sorted(fs)[:12]:
            print(f"   {os.path.join(r, f)}")
    raise FileNotFoundError(
        "manifest.json not found under /kaggle/input.\n"
        "Attach notebook 12's output: Add Input -> Your Work -> Notebook Output -> "
        "12_kaggle_artifact_import.")

ARTIFACT_DIR = os.path.dirname(_man)
MANIFEST = json.load(open(_man, encoding="utf-8"))
COLLECTION_NAME = MANIFEST["collection"]

SNAPSHOT_PATH = os.path.join(ARTIFACT_DIR, MANIFEST["snapshot"])
if not os.path.exists(SNAPSHOT_PATH):
    _cands = [os.path.join(ARTIFACT_DIR, f) for f in os.listdir(ARTIFACT_DIR)
              if not f.lower().endswith(".json")]
    if not _cands:
        raise FileNotFoundError(f"no snapshot file in {ARTIFACT_DIR}")
    SNAPSHOT_PATH = max(_cands, key=os.path.getsize)
    print(f"note: manifest says '{MANIFEST['snapshot']}', using "
          f"'{os.path.basename(SNAPSHOT_PATH)}' (largest file present)")

# ---- unzip the snapshot if it arrived as an archive -------------------------------------
# We check the first few bytes, not the file name. A file called .snapshot that is really a
# zip is the common case, because Drive keeps whatever name it was uploaded with.
# Careful: a real Qdrant snapshot IS a tar file, so "it is a tar" proves nothing. What marks
# a wrapper is holding exactly one .snapshot entry inside.
import zipfile, tarfile, gzip

def _is_zip(p):
    with open(p, "rb") as f:
        return f.read(4) in (b"PK\x03\x04", b"PK\x05\x06", b"PK\x07\x08")

def _is_gzip(p):
    with open(p, "rb") as f:
        return f.read(2) == b"\x1f\x8b"

def _is_tar(p):
    try:
        with open(p, "rb") as f:
            f.seek(257)
            return f.read(5) == b"ustar"
    except OSError:
        return False

def _tar_wrapper_member(p):
    try:
        with tarfile.open(p) as t:
            snaps = [m.name for m in t.getmembers()
                     if m.isfile() and m.name.lower().endswith(".snapshot")]
    except (tarfile.TarError, OSError):
        return None
    return snaps[0] if len(snaps) == 1 else None

def unwrap_snapshot(path, dest_dir):
    """Return a plain Qdrant snapshot, unpacking a zip/gzip/tar wrapper if there is one.
    Unpacks into dest_dir because /kaggle/input is read only."""
    os.makedirs(dest_dir, exist_ok=True)
    cur, peeled = path, []
    for _ in range(3):
        if _is_zip(cur):
            with zipfile.ZipFile(cur) as z:
                members = [i for i in z.infolist() if not i.is_dir()]
                if not members:
                    raise RuntimeError(f"{os.path.basename(cur)} is an empty zip")
                pick = next((i for i in members
                             if i.filename.lower().endswith(".snapshot")),
                            max(members, key=lambda i: i.file_size))
                need = pick.file_size / 1e9
                if disk_free_gb(dest_dir) < need + 0.5:
                    raise RuntimeError(f"need ~{need+0.5:.1f} GB to unzip, "
                                       f"{disk_free_gb(dest_dir):.1f} GB free")
                out = os.path.join(dest_dir, os.path.basename(pick.filename))
                print(f"   unzipping {pick.filename} ({need:.2f} GB) ...")
                t0 = time.time()
                with z.open(pick) as src, open(out, "wb") as dst:
                    shutil.copyfileobj(src, dst, 16 << 20)
                print(f"   done in {(time.time()-t0)/60:.1f} min")
            peeled.append("zip")
        elif _is_gzip(cur):
            base = os.path.basename(cur)
            for suf in (".tar.gz", ".tgz", ".gz"):
                if base.lower().endswith(suf):
                    base = base[: -len(suf)] + (".tar" if suf in (".tar.gz", ".tgz") else "")
                    break
            else:
                base += ".unpacked"
            out = os.path.join(dest_dir, base)
            print(f"   gunzipping {os.path.basename(cur)} ...")
            with gzip.open(cur, "rb") as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst, 16 << 20)
            peeled.append("gzip")
        else:
            member = _tar_wrapper_member(cur) if _is_tar(cur) else None
            if not member:
                break
            out = os.path.join(dest_dir, os.path.basename(member))
            print(f"   untarring {member} ...")
            with tarfile.open(cur) as t, open(out, "wb") as dst:
                shutil.copyfileobj(t.extractfile(member), dst, 16 << 20)
            peeled.append("tar")
        if cur != path and os.path.exists(cur):
            os.remove(cur)      # only ever remove files this function made
        cur = out
    if peeled:
        print(f"   unpacked: {' -> '.join(peeled)} -> {os.path.basename(cur)}")
    if _is_zip(cur) or _is_gzip(cur):
        raise RuntimeError(f"{os.path.basename(cur)} is still compressed after "
                           f"{len(peeled)} tries. Unpack it by hand and attach the plain file.")
    return cur

_kind = ("zip" if _is_zip(SNAPSHOT_PATH) else "gzip" if _is_gzip(SNAPSHOT_PATH)
         else "tar" if _is_tar(SNAPSHOT_PATH) else "unknown")
print(f"snapshot file type: {_kind}")
_orig = SNAPSHOT_PATH
SNAPSHOT_PATH = unwrap_snapshot(SNAPSHOT_PATH, WORK_ROOT)
SNAPSHOT_IS_EXTRACTED = SNAPSHOT_PATH != _orig

# ---- the KCC questions ------------------------------------------------------------------
POOL_PATH, KCC_RAW_PATH = None, None
for root in SEARCH_ROOTS:
    POOL_PATH = _glob1(os.path.join(root, "**", "kcc_pool.csv"))
    if POOL_PATH:
        break
if not POOL_PATH:
    for root in SEARCH_ROOTS:
        KCC_RAW_PATH = _glob1(os.path.join(root, "**", "kcc_qa_pairs.csv"))
        if KCC_RAW_PATH:
            break
    if not KCC_RAW_PATH:
        raise FileNotFoundError("neither kcc_pool.csv nor kcc_qa_pairs.csv was found")

print(f"\nartifacts : {ARTIFACT_DIR}")
print(f"snapshot  : {os.path.basename(SNAPSHOT_PATH)}  "
      f"{os.path.getsize(SNAPSHOT_PATH)/1e9:.2f} GB")
print(f"questions : {POOL_PATH or KCC_RAW_PATH}")
print("\nindex settings from manifest.json:")
for k in ("embed_model", "embed_dim", "max_seq_length", "query_prefix", "doc_prefix",
          "collection", "n_chunks", "tiers", "top_k_default"):
    print(f"   {k:<16} {MANIFEST.get(k)}")


STEP 1/14  Find the input files
expected ~0.2 min   |   about 71 min left of ~71 min total
snapshot file type: tar

artifacts : /kaggle/input/notebooks/harlivsingh1122777/distil-real/rag_production_bge_m3
snapshot  : agri_knowledge-79188001122225-2026-07-27-13-03-44.snapshot  3.80 GB
questions : /kaggle/input/notebooks/harlivsingh1122777/distil-real/kcc_pool.csv

index settings from manifest.json:
   embed_model      BAAI/bge-m3
   embed_dim        1024
   max_seq_length   512
   query_prefix     
   doc_prefix       
   collection       agri_knowledge
   n_chunks         723439
   tiers            {'fallback': 0.56, 'grounded': 0.66}
   top_k_default    5


## 2. Start Qdrant

Qdrant must run as a server. Its built-in local mode ignores the search index and scans every
record, which turns a 40 ms search into about 25 seconds. It never reports an error — it is
just slow. Local mode also cannot load a snapshot at all.

In [5]:
step(2, "Start the Qdrant server")
import subprocess, tarfile, requests

QDRANT_STORAGE = os.path.join(WORK_ROOT, "qdrant_storage")
QDRANT_SNAPDIR = os.path.join(WORK_ROOT, "qdrant_snapshots")
QDRANT_BIN     = os.path.join(WORK_ROOT, "qdrant")
os.makedirs(QDRANT_STORAGE, exist_ok=True)
os.makedirs(QDRANT_SNAPDIR, exist_ok=True)

def qdrant_alive(url=QDRANT_URL, timeout=1):
    try:
        return requests.get(f"{url}/readyz", timeout=timeout).ok
    except Exception:
        return False

def binary_ok(path=QDRANT_BIN):
    if not os.path.exists(path):
        return False
    try:
        return subprocess.run([path, "--version"], capture_output=True,
                              timeout=60).returncode == 0
    except Exception:
        return False

qdrant_proc = None
if qdrant_alive():
    print("Qdrant is already running")
else:
    if not binary_ok():
        if os.path.exists(QDRANT_BIN):
            os.remove(QDRANT_BIN)
        rel = requests.get("https://api.github.com/repos/qdrant/qdrant/releases/latest",
                           timeout=30).json()
        asset = None
        # musl build first: it has no C library dependency. The gnu build downloads fine and
        # then fails to start, so checking that the file exists is not enough.
        for suffix in ("x86_64-unknown-linux-musl.tar.gz",
                       "x86_64-unknown-linux-gnu.tar.gz"):
            asset = next((a for a in rel["assets"] if a["name"].endswith(suffix)), None)
            if asset:
                break
        if asset is None:
            raise RuntimeError("no linux build in release " + rel["tag_name"])
        print(f"downloading qdrant {rel['tag_name']} ...")
        tgz = os.path.join(WORK_ROOT, "q.tar.gz")
        with open(tgz, "wb") as f:
            f.write(requests.get(asset["browser_download_url"], timeout=600).content)
        with tarfile.open(tgz) as t:
            try:
                t.extractall(WORK_ROOT, filter="data")
            except TypeError:
                t.extractall(WORK_ROOT)
        os.chmod(QDRANT_BIN, 0o755)
        os.remove(tgz)
        if not binary_ok():
            raise RuntimeError("the downloaded qdrant will not run here")

    qdrant_log = os.path.join(WORK_ROOT, "qdrant.log")
    env = dict(os.environ,
               QDRANT__STORAGE__STORAGE_PATH=QDRANT_STORAGE,
               QDRANT__STORAGE__SNAPSHOTS_PATH=QDRANT_SNAPDIR,
               QDRANT__TELEMETRY_DISABLED="true")
    qdrant_proc = subprocess.Popen([QDRANT_BIN], env=env, cwd=WORK_ROOT,
                                   stdout=open(qdrant_log, "w"), stderr=subprocess.STDOUT)
    for _ in range(120):
        if qdrant_alive():
            break
        if qdrant_proc.poll() is not None:
            raise RuntimeError(f"qdrant stopped early. Log:\n"
                               + open(qdrant_log).read()[-1500:])
        time.sleep(1)
    else:
        raise RuntimeError(f"qdrant did not start in 120s -- see {qdrant_log}")
    print("Qdrant is ready on port 6333")


--- step 1 done in 0.0 min (total so far 0.0 min) ---

STEP 2/14  Start the Qdrant server
expected ~1.0 min   |   about 71 min left of ~71 min total
downloading qdrant v1.18.3 ...
Qdrant is ready on port 6333


## 3. Load the search index

Two ways to load, tried in order. The first reads the file straight from disk. The second
uploads it over HTTP, which works but builds the whole 3.9 GB request in memory first.

This is usually the slowest step after the teacher. Expect 5–10 minutes.

In [6]:
step(3, "Load the search index")
from qdrant_client import QdrantClient

_existing = requests.get(f"{QDRANT_URL}/collections", timeout=60).json()["result"]["collections"]
if any(c["name"] == COLLECTION_NAME for c in _existing):
    print(f"collection '{COLLECTION_NAME}' is already loaded, skipping")
else:
    gb = os.path.getsize(SNAPSHOT_PATH) / 1e9
    print(f"loading {gb:.2f} GB ... (this is the slow one)")
    t0, ok = time.time(), False
    try:
        r = requests.put(
            f"{QDRANT_URL}/collections/{COLLECTION_NAME}/snapshots/recover?wait=true",
            json={"location": "file://" + os.path.abspath(SNAPSHOT_PATH)}, timeout=7200)
        ok = r.ok and r.json().get("result") is True
        if not ok:
            print(f"   reading from disk was refused ({r.status_code}), uploading instead")
    except Exception as e:
        print(f"   reading from disk failed ({type(e).__name__}), uploading instead")
    if not ok:
        with open(SNAPSHOT_PATH, "rb") as fh:
            r = requests.post(
                f"{QDRANT_URL}/collections/{COLLECTION_NAME}/snapshots/upload?priority=snapshot",
                files={"snapshot": (os.path.basename(SNAPSHOT_PATH), fh)}, timeout=7200)
        r.raise_for_status()
    print(f"loaded in {(time.time()-t0)/60:.1f} min")

qdrant_client = QdrantClient(url=QDRANT_URL, timeout=600)
info = qdrant_client.get_collection(COLLECTION_NAME)
print(f"\ncollection '{COLLECTION_NAME}'")
print(f"   points  : {info.points_count:,}")
print(f"   indexed : {info.indexed_vectors_count:,}")
print(f"   status  : {info.status}")

if info.points_count < 100_000:
    raise RuntimeError(
        f"only {info.points_count:,} points loaded. The full index has about 723,000. "
        "A partial load answers every question from a fraction of the data without "
        "reporting an error.")

# The copy we unpacked is no longer needed once the index is live, and the teacher still
# needs disk space later.
if SNAPSHOT_IS_EXTRACTED and os.path.exists(SNAPSHOT_PATH):
    _gb = os.path.getsize(SNAPSHOT_PATH) / 1e9
    os.remove(SNAPSHOT_PATH)
    print(f"removed the unpacked copy, {_gb:.2f} GB freed "
          f"({disk_free_gb():.1f} GB free now)")
vram_report("index loaded")


--- step 2 done in 0.0 min (total so far 0.1 min) ---

STEP 3/14  Load the search index
expected ~9.0 min   |   about 70 min left of ~71 min total
loading 3.80 GB ... (this is the slow one)
   reading from disk was refused (403), uploading instead
loaded in 1.2 min

collection 'agri_knowledge'
   points  : 723,439
   indexed : 723,439
   status  : yellow
[index loaded] gpu0 15.5 GB free   gpu1 15.5 GB free   disk 13.2 GB free


## 4. Load the embedding model

Every setting comes from `manifest.json`. Getting one wrong fails quietly: a wrong prefix makes
every vector slightly wrong, and a wrong threshold refuses every question including perfect
matches. The vector size check is the one that can fail loudly, so it is an assert.

In [7]:
step(4, "Load the embedding model")
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = MANIFEST["embed_model"]
MODEL_MAX_TOKENS = MANIFEST["max_seq_length"]
QUERY_PREFIX     = MANIFEST["query_prefix"]
DOC_PREFIX       = MANIFEST["doc_prefix"]
TOP_K_DEFAULT    = MANIFEST["top_k_default"]
TIER_GROUNDED    = MANIFEST["tiers"]["grounded"]
TIER_FALLBACK    = MANIFEST["tiers"]["fallback"]
FUSION_WEIGHTS   = MANIFEST["fusion_weights"]

embed_query = lambda t: QUERY_PREFIX + t

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"loading {EMBED_MODEL_NAME} on {device} ...")
t0 = time.time()
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
embed_model.max_seq_length = MODEL_MAX_TOKENS

EMBED_DIM = embed_model.get_sentence_embedding_dimension()
assert EMBED_DIM == MANIFEST["embed_dim"], (
    f"vector size mismatch: model gives {EMBED_DIM}, index was built with "
    f"{MANIFEST['embed_dim']}. This is the wrong model for this index.")

print(f"ready in {(time.time()-t0)/60:.1f} min, vector size {EMBED_DIM}")
print(f"score bands: refuse below {TIER_FALLBACK}, "
      f"answer with warning up to {TIER_GROUNDED}, answer above that")
vram_report("embedder loaded")


--- step 3 done in 1.3 min (total so far 1.3 min) ---

STEP 4/14  Load the embedding model
expected ~2.0 min   |   about 61 min left of ~71 min total
loading BAAI/bge-m3 on cuda ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

ready in 0.4 min, vector size 1024
score bands: refuse below 0.56, answer with warning up to 0.66, answer above that
[embedder loaded] gpu0 13.2 GB free   gpu1 15.5 GB free   disk 12.2 GB free


/tmp/ipykernel_24/2056485531.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = embed_model.get_sentence_embedding_dimension()


## 5. Crop and district name cleanup

KCC writes crop names like `Paddy (Dhan)` and `Bengal Gram (Gram/Chick Pea)`. 63 of 281 crop
names have brackets. The first version of this converter only matched plain names, so a filter
for `rice` matched **0 of 112,269 rice records** — with no error, because an empty result looks
exactly like "nothing found".

So we try the full name, then the part before the bracket, then each name inside it.

In [8]:
step(5, "Set up name cleanup")

DISTRICT_CANON = {
    "allahabad": "prayagraj", "faizabad": "ayodhya",
    "prabuddh nagar": "shamli", "prabudh nagar": "shamli",
    "bhim nagar": "sambhal", "panchsheel nagar": "hapur",
    "jyotiba phule nagar": "amroha", "jyotibaphule nagar": "amroha",
    "kanshi ram nagar": "kasganj", "kanshiram nagar": "kasganj",
    "chhatrapati shahuji maharaj nagar": "amethi",
    "mahamaya nagar": "hathras", "ramabai nagar": "kanpur dehat",
    "banaras": "varanasi", "kashi": "varanasi",
    "kanpur city": "kanpur nagar", "maharahganj": "maharajganj",
    "sant ravidas nagar": "bhadohi",
}

def canon_district(raw):
    if not raw or str(raw).lower() in ("unknown", "nan", "none", ""):
        return None
    d = re.sub(r"\s+", " ", str(raw).strip().lower())
    return DISTRICT_CANON.get(d, d)

CROP_CANON = {
    "rice": "rice", "paddy": "rice", "dhan": "rice", "chawal": "rice",
    "wheat": "wheat", "gehun": "wheat", "gehu": "wheat", "kanak": "wheat",
    "maize": "maize", "makka": "maize", "makai": "maize", "bhutta": "maize", "corn": "maize",
    "\u092e\u0915\u094d\u0915\u0947": "maize",
    "sugarcane": "sugarcane", "ganna": "sugarcane", "ganne": "sugarcane",
    "noble cane": "sugarcane", "\u0917\u0928\u094d\u0928\u0947": "sugarcane",
    "potato": "potato", "aloo": "potato", "alu": "potato",
    "mustard": "mustard", "sarson": "mustard", "raya": "mustard",
    "indian mustard": "mustard", "indian rapeseed and mustard": "mustard",
    "yellow sarson": "mustard",
    "urad": "urad", "black gram": "urad", "urd": "urad", "urd bean": "urad",
    "gram": "gram", "bengal gram": "gram", "chana": "gram", "chane": "gram",
    "chick pea": "gram", "kabuli": "gram",
    "moong": "moong", "green gram": "moong", "moong bean": "moong", "mung": "moong",
    "arhar": "arhar", "pigeon pea": "arhar", "red gram": "arhar", "tur": "arhar",
    "masur": "masur", "lentil": "masur",
    "okra": "okra", "bhindi": "okra", "ladysfinger": "okra",
    "bajra": "bajra", "pearl millet": "bajra", "bulrush millet": "bajra",
    "spiked millet": "bajra",
    "jowar": "jowar", "sorghum": "jowar", "great millet": "jowar",
    "barley": "barley", "jau": "barley",
    "sesame": "sesame", "til": "sesame", "gingelly": "sesame", "sesamum": "sesame",
    "groundnut": "groundnut", "pea nut": "groundnut", "peanut": "groundnut",
    "mung phalli": "groundnut",
    "colocasia": "arvi", "arvi": "arvi", "arbi": "arvi", "arum": "arvi",
    "cotton": "cotton", "kapas": "cotton",
    "soybean": "soybean", "bhat": "soybean",
    "linseed": "linseed", "alsi": "linseed",
    "spinach": "spinach", "palak": "spinach",
    "methi": "fenugreek", "fenugreek": "fenugreek",
    "rajma": "rajma", "french bean": "rajma",
    "sunflower": "sunflower", "suryamukhi": "sunflower",
    "finger millet": "ragi", "fingermillet": "ragi", "ragi": "ragi", "mandika": "ragi",
    "pea": "pea", "peas": "pea", "matar": "pea", "field peas": "pea", "garden peas": "pea",
}

_CROP_PAREN = re.compile(r"^([^(]+?)\s*\((.*)\)\s*$")
_CROP_NULLS = ("unknown", "nan", "none", "", "na", "n/a", "other", "others")

def canon_crop(raw):
    if raw is None:
        return None
    c = re.sub(r"\s+", " ", str(raw).strip().lower())
    if c in _CROP_NULLS:
        return None
    if c in CROP_CANON:
        return CROP_CANON[c]
    m = _CROP_PAREN.match(c)
    if m:
        base, inner = m.group(1).strip(), m.group(2)
        if base in CROP_CANON:
            return CROP_CANON[base]
        for alias in re.split(r"[/,]", inner):
            if alias.strip() in CROP_CANON:
                return CROP_CANON[alias.strip()]
        return base
    return c

# Running it twice must give the same answer, otherwise the stored value and the search
# filter would disagree.
for probe in ("Paddy (Dhan)", "Bengal Gram (Gram/Chick Pea)", "rice", "Maize (Makka)"):
    once = canon_crop(probe)
    assert canon_crop(once) == once, f"canon_crop is not stable for {probe!r}"
print("name cleanup ready and stable")


--- step 4 done in 1.0 min (total so far 2.4 min) ---

STEP 5/14  Set up name cleanup
expected ~0.1 min   |   about 59 min left of ~71 min total
name cleanup ready and stable


## 6. The search function

Copied from `10_rag_generation.ipynb`. The training prompt has to look exactly like the
serving prompt, so we use the same function rather than writing a new one.

The confidence band is worked out from the **raw** score, never the weighted one. Weighting is
only there to reorder results; using it for the threshold would let a multiplier create
confidence the match does not have.

In [9]:
step(6, "Set up the search function")
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

def _citation(p):
    if p.get("source_type") == "pdf":
        return {"corpus": "pdf", "file": p.get("filename"),
                "pages": [p.get("page_start"), p.get("page_end")],
                "section": p.get("heading_hierarchy") or None,
                "doc_category": p.get("doc_category"),
                "district": p.get("district"), "year": p.get("year")}
    return {"corpus": "kcc", "record": "KCC Q&A", "crop": p.get("crop"),
            "district": p.get("district"), "season": p.get("season"),
            "query_type": p.get("query_type"), "year": p.get("year")}


def _search_core(query, top_k=None, intent="general", source_type=None,
                 doc_category=None, query_type=None, crop=None, district=None,
                 season=None, language=None, year_from=None, only_tables=None,
                 fusion_override=None, qvec=None):
    top_k   = TOP_K_DEFAULT if top_k is None else top_k
    wtable  = fusion_override if fusion_override is not None else FUSION_WEIGHTS
    weights = wtable.get(intent, wtable.get("general", {}))
    timing  = {"embed_ms": 0.0, "search_ms": 0.0}

    def sub_search(stype, vec):
        must = [FieldCondition(key="source_type", match=MatchValue(value=stype))]
        if doc_category: must.append(FieldCondition(key="doc_category", match=MatchValue(value=doc_category)))
        if query_type:   must.append(FieldCondition(key="query_type", match=MatchValue(value=query_type)))
        if crop:         must.append(FieldCondition(key="crop", match=MatchValue(value=canon_crop(crop))))
        if district:     must.append(FieldCondition(key="district", match=MatchValue(value=canon_district(district))))
        if season:       must.append(FieldCondition(key="season", match=MatchValue(value=season)))
        if language:     must.append(FieldCondition(key="language", match=MatchValue(value=language)))
        if year_from:    must.append(FieldCondition(key="year", range=Range(gte=year_from)))
        if only_tables:  must.append(FieldCondition(key="has_table", match=MatchValue(value=True)))
        return qdrant_client.query_points(
            collection_name=COLLECTION_NAME, query=vec,
            query_filter=Filter(must=must), limit=top_k, with_payload=True).points

    try:
        t0 = time.time()
        if qvec is None:
            qvec = embed_model.encode(embed_query(query), normalize_embeddings=True).tolist()
        timing["embed_ms"] = (time.time() - t0) * 1000

        t0 = time.time()
        sources = [source_type] if source_type else ["pdf", "kcc"]
        hits = []
        for stype in sources:
            for h in sub_search(stype, qvec):
                hits.append({
                    "raw_score": round(float(h.score), 4),
                    "fused_score": round(float(h.score) * weights.get(stype, 1.0), 4),
                    "text": h.payload.get("text", ""),
                    "source_type": stype,
                    "has_table": bool(h.payload.get("has_table", False)),
                    "chunk_id": h.payload.get("chunk_id"),
                    "citation": _citation(h.payload),
                })
        hits.sort(key=lambda x: x["fused_score"], reverse=True)
        hits = hits[:top_k]
        timing["search_ms"] = (time.time() - t0) * 1000
    except Exception as e:
        # Search never raises. A failure comes back as tier "error" so the caller's loop
        # keeps going.
        return {"query": query, "tier": "error", "top_score": 0.0,
                "results": [], "error": str(e), "timing": timing}

    best_raw = max((h["raw_score"] for h in hits), default=0.0)
    tier = ("grounded" if best_raw >= TIER_GROUNDED
            else "fallback_with_disclaimer" if best_raw >= TIER_FALLBACK
            else "abstain_out_of_scope")
    return {"query": query, "intent": intent, "tier": tier,
            "top_score": round(best_raw, 4), "results": hits, "timing": timing}


def encode_queries(queries, batch_size=64):
    """Encode many questions in one GPU pass instead of one call each."""
    import numpy as np
    vecs = embed_model.encode([embed_query(q) for q in queries],
                              normalize_embeddings=True, batch_size=batch_size,
                              show_progress_bar=True, convert_to_numpy=True)
    return [v.tolist() for v in np.asarray(vecs)]


_s = _search_core("interest subvention on crop loans", top_k=3, intent="policy")
print(f"test search: tier={_s['tier']} top_score={_s['top_score']} "
      f"hits={len(_s['results'])} "
      f"({_s['timing']['embed_ms']:.0f} ms embed + {_s['timing']['search_ms']:.0f} ms search)")
assert _s["tier"] != "error", f"search is broken: {_s.get('error')}"


--- step 5 done in 0.0 min (total so far 2.4 min) ---

STEP 6/14  Set up the search function
expected ~0.2 min   |   about 59 min left of ~71 min total
test search: tier=grounded top_score=0.7546 hits=3 (997 ms embed + 131 ms search)


## 7. Prompts, language detection, and scorers

### Language detection has three answers now
`hi` for Devanagari, `hinglish` for Hindi words written in Roman letters, and `en`. The old
version only checked for Devanagari, so Hinglish came out as English — and then we would tell
the model to answer a Hinglish question in English, which breaks our own rule 3.

### The prompt names the answer language at the very end
Retrieved chunks are mostly Hindi. Without a clear instruction the model copies their language.
So the last line before the model starts writing says which language to use, and one rule says
not to copy the language of the context.

### `numeric_grounding` was rebuilt and is checked before use
The old version had no idea what a thousands separator is. Context saying `Rs 6,000` became
`['6','000']` while an answer saying `6000` became `['6000']`, so a number that was there word
for word came out as unsupported. It also counted list markers (`1.`, `2.`) as quantities.

That matters more here than in a report, because this scorer **decides which teacher answers we
keep**. A scorer with false alarms quietly throws away good rows. The cell below checks it
against known answers first.

What it still cannot do is spot a wrong meaning. An answer that mixes up two similar scheme
names passes it. That is why every row keeps the KCC expert answer beside it for review.

In [10]:
step(7, "Set up prompts and scorers")
from transformers import AutoTokenizer

gen_tok = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, token=os.environ.get("HF_TOKEN"))
print(f"prompt tokenizer: {STUDENT_MODEL_ID}")

SYSTEM_RULES = (
    "You are an agricultural advisory assistant for farmers in Uttar Pradesh, India.\n"
    "RULES:\n"
    "1. Answer ONLY from the CONTENT of the numbered context items below. Never use outside knowledge.\n"
    "\n"
    "2. STRICTLY FILTER THE CONTEXT. Only use items that explicitly mention the EXACT crop, scheme, or disease/pest asked in the question.\n"
    "   - If the question asks about a specific government scheme (e.g., PM-Kisan), do NOT use data from a differently named but similar scheme (e.g., PM-Kisan Maan-Dhan).\n"
    "   - If the question asks about a crop (e.g., wheat), do NOT use data about another crop (e.g., paddy), even if the problem sounds similar.\n"
    "   - If the question asks about a specific disease (e.g., yellow rust), do NOT use data about a different disease (e.g., smut).\n"
    "   - If you have to choose between using a loosely related item or refusing, ALWAYS refuse. Safety over completeness.\n"
    "\n"
    "3. IF THE FILTERED CONTENT DOES NOT EXACTLY ANSWER THE QUESTION, your whole answer is ONE short sentence saying you do not have that information. Then stop. Do NOT describe what the content is about. Do NOT offer other topics. Do NOT say what else is available. Do NOT tell the farmer to ask about something else. The refusal IS the whole answer.\n"
    "\n"
    "4. Never invent a dosage, price, date, eligibility rule, or scheme benefit.\n"
    "\n"
    "5. Speak directly to the farmer and tell them what to DO.\n"
    "\n"
    "6. NEVER mention district names, years, file names or record numbers in your answer. The SOURCE line above each item is only for the citation at the end.\n"
    "\n"
    "7. Do NOT put citation numbers inside your sentences. Write the whole answer first. Then put the citations on the last line only.\n"
    "\n"
    "8. The context is mostly in Hindi. Do NOT copy its language. The language you must write in is stated at the very end of this message.\n"
    "\n"
    "9. Be concise and practical: 80-120 tokens, no preamble, no repetition.\n"
    "\n"
    "10. CRITICAL – SCHEME/CROP DISAMBIGUATION: If two schemes or crops have similar names, treat them as completely different. Do not transfer information between them. If you cannot clearly distinguish which one the question refers to, refuse.\n"
    "\n"
    "11. If the context contains information about a different scheme, crop, or disease than the one asked, you must explicitly mention the mismatch in your refusal. Say: 'The context provides information about [different scheme/crop], but your question asks about [asked scheme/crop]. I do not have information about [asked].' This clarifies the confusion without guessing, keeps the farmer informed, and still counts as a proper refusal."
)
OUTPUT_FORMAT = (
    "FORMAT: Write one short paragraph of practical advice with NO bracket numbers in it. "
    "Then write a final line, exactly: Sources: [1], [2]\n\n"
    "Some good and bad examples for you response are:\n\n"
    "GOOD (content answers the question):\n"
    "  Spray mancozeb 2 g per litre of water at the first sign of disease.\n"
    "  Sources: [1], [3]\n\n"
    "GOOD (content does not answer the question):\n"
    "  I do not have information about that.\n"
    "  Sources: [1]\n\n"
    "GOOD (context is about a different scheme/crop than the question, lets say farmer asked about PM Kisan but context doesnt mention that scheme but mentions another one.):\n"
    "  I do not have information about PM-Kisan.\n"
    "  Sources: [1], [2]\n\n"
    "BAD (describes what the content is about):\n"
    "  Information for Budaun district [1] and Aligarh [2] is available.\n\n"
    "BAD (refuses, then offers other topics):\n"
    "  I do not have information about plant protection. I can provide information "
    "about cultivation and sowing times.\n\n"
    "BAD (tells the farmer to ask about something else):\n"
    "  Ask about fenugreek. Ask about sugarcane."

)

LANG_NAME = {
    "hi":       "Hindi (Devanagari script)",
    "en":       "English",
    "hinglish": "Hinglish (Hindi words written in Roman/Latin letters)",
}

# Common Hindi words that farmers type in Roman letters. Used to tell Hinglish from English,
# which cannot be done by looking at the script because both use the same letters.
HINGLISH_WORDS = {
    "kya", "kaise", "kaisa", "hai", "hain", "ho", "me", "mein", "ka", "ki", "ke", "ko", "se",
    "par", "kare", "karein", "karu", "karna", "kaun", "kaunsa", "koinsi", "bataye", "batao",
    "lag", "gaya", "gaye", "rahi", "raha", "rahe", "nahi", "nahin", "hota", "hoti", "liye",
    "bare", "sabse", "sbse", "achha", "accha", "kitna", "kitni", "kab", "kahan", "kyun",
    "chahiye", "milega", "dalna", "dale", "lagta", "krna", "kr",
    "fasal", "kisan", "khet", "dawa", "dawai", "beej", "khad", "urvarak", "paani", "pani",
    "rog", "kide", "keeda", "keede", "upchar", "ilaj", "bhav", "mandi", "nursery", "patti",
    "pila", "peela", "ratua", "jhulsa", "sadan", "kharab",
}


def deva_ratio(s):
    """Share of letters that are Devanagari."""
    letters = [ch for ch in s if ch.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for ch in letters if "\u0900" <= ch <= "\u097f") / len(letters)


def detect_lang(text):
    """Return 'hi', 'hinglish' or 'en'."""
    if deva_ratio(text) > 0.4:
        return "hi"
    words = re.findall(r"[a-z]+", str(text).lower())
    if words:
        hits = sum(1 for w in words if w in HINGLISH_WORDS)
        if hits >= 2 and hits / len(words) >= 0.20:
            return "hinglish"
    return "en"


def format_context(hits, ctx_top_k=5, char_cap=700):
    lines, used = [], []
    for i, h in enumerate(hits[:ctx_top_k], 1):
        c = h["citation"]
        if c["corpus"] == "pdf":
            pages = c.get("pages") or [None, None]
            where = f"PDF {c.get('file')} p.{pages[0]}"
        else:
            where = f"KCC record, crop={c.get('crop')}, district={c.get('district')}, "\
                    f"year={c.get('year')}"
        txt = re.sub(r"\s+", " ", h["text"]).strip()[:char_cap]
        # The source label is on its own line and clearly named. In the first real run the
        # model read this label as content and produced answers like "information is
        # available for Budaun district [1] and Aligarh [2]" -- listing districts and years
        # that appear nowhere except here. Labelling it and telling the model to ignore it
        # (rule 4) is the fix.
        lines.append(f"[{i}] SOURCE (for citation only, do not repeat): {where}\n"
                     f"    CONTENT: {txt}")
        used.append({"n": i, "chunk_id": h["chunk_id"], "source_type": h["source_type"],
                     "raw_score": h["raw_score"], "citation": c, "text": txt})
    return "\n\n".join(lines), used


def build_prompt(query, hits, tier, ctx_top_k=5, tool_data=None, char_cap=700, lang=None):
    """Build the prompt. The answer language goes last, right before the model writes."""
    ctx, used = format_context(hits, ctx_top_k=ctx_top_k, char_cap=char_cap)
    tool_block = json.dumps(tool_data, ensure_ascii=False) if tool_data else "(none available)"
    lang = lang or detect_lang(query)
    body = (f"{SYSTEM_RULES}\n"
            f"RELEVANCE TIER: {tier}\n\n"
            f"CONTEXT:\n{ctx}\n\n"
            f"STRUCTURED TOOL DATA:\n{tool_block}\n\n"
            f"FARMER'S QUESTION:\n{query}\n\n"
            f"{OUTPUT_FORMAT}\n\n"
            f"### WRITE YOUR ENTIRE ANSWER IN {LANG_NAME[lang].upper()}. ###\n"
            f"The context above is mostly Hindi, but you must answer in {LANG_NAME[lang]}.")
    try:
        prompt = gen_tok.apply_chat_template([{"role": "user", "content": body}],
                                             tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = f"<start_of_turn>user\n{body}<end_of_turn>\n<start_of_turn>model\n"
    return prompt, used


# ---- scorers ---------------------------------------------------------------------------
# Grouped form first, so "6,000" matches as one number instead of splitting into "6" and "000".
NUM_RE  = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?")
CITE_RE = re.compile(r"\[\s*\d+(?:\s*,\s*\d+)*\s*\]")
# "1." or "2)" at the start of a line is a list marker, not an amount.
LIST_MARKER_RE = re.compile(r"(?:(?<=\n)|(?<=^)|(?<=;))\s*\d{1,2}\s*[.)\]:]", re.M)


def _norm_num(tok):
    """'6,000' -> '6000', '6000.0' -> '6000'. Same form on both sides of the comparison."""
    try:
        return f"{float(tok.replace(',', '')):g}"
    except ValueError:
        return None


def _numbers(text):
    if not text:
        return []
    t = LIST_MARKER_RE.sub(" ", CITE_RE.sub(" ", text))
    return [n for n in (_norm_num(m) for m in NUM_RE.findall(t)) if n is not None]


def numeric_grounding(answer_text, context_text):
    """How many of the answer's numbers appear in the context. 1.0 if it states no numbers."""
    ctx_nums = set(_numbers(context_text))
    ans_nums = _numbers(answer_text)
    if not ans_nums:
        return 1.0, []
    unsupported = [n for n in ans_nums if n not in ctx_nums]
    return 1.0 - len(unsupported) / len(ans_nums), unsupported


def language_match(query, answer_text):
    """1.0 if the answer is in the language we asked for."""
    if not answer_text:
        return None
    want, got = detect_lang(query), detect_lang(answer_text)
    if want == "hi":
        return 1.0 if got == "hi" else 0.0
    # English and Hinglish both use Roman letters, so the test is "not Devanagari".
    return 1.0 if got != "hi" else 0.0


def has_citation(answer_text):
    return bool(answer_text and CITE_RE.search(answer_text))


# Words that show the model declined instead of guessing. Used to keep only the refusal
# rows where the teacher actually refused -- a teacher that answers a question it cannot
# support would teach the student to do the same.
REFUSAL_WORDS = (
    # "do not have" first: it is how the teacher actually phrases a refusal most of the
    # time ("I do not have information about X"). The first version of this list missed it
    # entirely, so every clean refusal would have been thrown away as "the teacher did not
    # refuse" -- losing exactly the rows we are trying to collect.
    "do not have", "don't have", "dont have", "no further information",
    "not available", "not provided", "does not contain", "doesn't contain", "no information",
    "not mentioned", "cannot answer", "can't answer", "outside", "not in the context",
    "unable to", "not specified", "no data", "cannot advise", "can't advise",
    "uplabdh nahi", "jankari nahi", "nahi hai", "bahar hai",
    "\u0928\u0939\u0940\u0902 \u0939\u0948", "\u0909\u092a\u0932\u092c\u094d\u0927 \u0928\u0939\u0940\u0902",
    "\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u0928\u0939\u0940\u0902", "\u092c\u093e\u0939\u0930 \u0939\u0948",
)


def looks_like_refusal(text):
    """True if the answer declines rather than guessing."""
    if not text:
        return False
    low = text.lower()
    return any(w in low for w in REFUSAL_WORDS)


# Answers that describe where information lives instead of giving advice. Seen a lot in the
# first real run: "information about bakani disease is available for Budaun, Shamli,
# Aligarh...". Training on these teaches the student to summarise the index, not to help.
META_PHRASES = (
    "is available", "are available", "information is", "data is available",
    "\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u0909\u092a\u0932\u092c\u094d\u0927 \u0939\u0948",
    "\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u092e\u094c\u091c\u0942\u0926 \u0939\u0948",
    "\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u0915\u0947 \u0932\u093f\u090f",
    "\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u0926\u0940 \u0917\u0908 \u0939\u0948",
    "jankari uplabdh hai", "jankari maujood hai", "jankari ke liye",
)


# Phrases that offer other topics. These are wrong even inside a refusal, which is the
# case the first filter missed: "I do not have information about plant protection for
# Brinjal. I can provide information about cultivation and sowing times." The refusal check
# used to short-circuit and let the whole thing through, so the student would have learned
# to answer a maize question by offering mango information.
OFFER_PHRASES = (
    "i can provide", "i can give", "i can offer", "i can tell you about",
    "i have information", "find information about", "you can inquire", "you can ask",
    "you may ask", "you can also find", "consult information", "ask about",
    "is available", "are available", "available records", "records discuss",
    "context discusses", "the provided context", "please ask",
    "\u0926\u0947 \u0938\u0915\u0924\u093e",           # de sakta (can give)
    "\u092a\u0942\u091b \u0938\u0915\u0924\u0947",     # puchh sakte (can ask)
    "\u092a\u0942\u091b\u0947\u0902",                     # puchhein (ask)
    "\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u0909\u092a\u0932\u092c\u094d\u0927 \u0939\u0948",
)


def offers_alternatives(text):
    """True if the answer points the farmer at some other topic."""
    if not text:
        return False
    low = text.lower()
    return any(w in low for w in OFFER_PHRASES)


# Crop words in Devanagari. CROP_CANON only holds Roman spellings, and the stitched answers
# are usually Hindi, so the counter below needs both.
HINDI_CROP_WORDS = {
    "गेहूं": "wheat", "धान": "rice",
    "चावल": "rice", "आलू": "potato",
    "गन्ना": "sugarcane", "गन्ने": "sugarcane",
    "मक्का": "maize", "चना": "gram",
    "मटर": "pea", "प्याज": "onion",
    "टमाटर": "tomato", "गोभी": "cauliflower",
    "भिंडी": "okra", "खीरा": "cucumber",
    "खीरे": "cucumber", "सरसों": "mustard",
    "बाजरा": "bajra", "लहसुन": "garlic",
    "मूली": "radish", "मसूर": "masur",
    "उरद": "urad", "सोयाबीन": "soybean",
    "आम": "mango", "केला": "banana",
    "मिर्च": "chilli", "तिल": "sesame",
    "जौ": "barley", "मूंग": "moong",
}


def count_crops(text):
    """How many different crops the answer talks about."""
    if not text:
        return 0
    found = set()
    low = text.lower()
    for word in re.findall(r"[a-z]+", low):
        if word in CROP_CANON:
            found.add(CROP_CANON[word])
    for word, canon in HINDI_CROP_WORDS.items():
        if word in text:
            found.add(canon)
    return len(found)


def mixes_crops(text, limit=2):
    """True if the answer covers more crops than any single question could need.

    Seen in the smoke run on vague questions like "Asked about plant protection": search
    returns five unrelated chunks and the model stitches them into one answer covering
    garlic, bajra and mustard. No phrase catches that, but the crop count does. It also
    matters for safety -- mixing dose advice across crops is how a farmer gets the wrong
    chemical."""
    return count_crops(text) > limit


def looks_like_meta_answer(text):
    """True if the answer talks about the context instead of answering the question.

    Offering other topics is checked FIRST, because it is wrong whether or not the answer
    also refuses. Only after that does a plain refusal get a pass -- "I do not have that
    information" on its own is exactly the behaviour we want to teach."""
    if not text:
        return False
    if offers_alternatives(text):
        return True
    if looks_like_refusal(text):
        return False
    low = text.lower()
    return any(pz.lower() in low for pz in META_PHRASES)


def has_inline_citation(text):
    """True if a [n] marker appears before the final Sources line.

    Rule 5 says the answer text must be clean and citations go on the last line only. The
    first real run put them mid-sentence constantly, which makes the answer hard to read
    and hard to check."""
    if not text:
        return False
    low = text.lower()
    cut = low.rfind("sources:")
    body = text[:cut] if cut != -1 else text
    return bool(CITE_RE.search(body))


# ---- check the scorer before we filter anything with it --------------------------------
_CASES = [
    ("Rs 6,000 per year",                              "Rs 6000 per year",                   1.0, []),
    ("Rs 6000 per year",                               "Rs 6000 per year",                   1.0, []),
    ("Rs. 2,000 each in 3 instalments totalling 6,000","3 instalments of 2000, total 6000",  1.0, []),
    ("apply urea 30 kg per acre",                      "1. plough\n2. apply urea 30 kg [1]", 1.0, []),
    ("apply urea 30 kg per acre",                      "spray 45 kg of urea [1]",            0.0, ["45"]),
    ("apply urea 30 kg per acre in 2 splits",          "30 kg urea in 3 splits [1]",         0.5, ["3"]),
    ("apply urea 1,200 kg per hectare",                "use 2,500 kg per hectare [1]",       0.0, ["2500"]),
    ("no figures here",                                "no figures in the answer either",    1.0, []),
]
for ctx, ans, want_score, want_unsup in _CASES:
    got, unsup = numeric_grounding(ans, ctx)
    assert abs(got - want_score) < 1e-9 and unsup == want_unsup, (
        f"numeric_grounding check FAILED\n  context: {ctx!r}\n  answer : {ans!r}\n"
        f"  wanted : {want_score} {want_unsup}\n  got    : {got} {unsup}")
print(f"numeric_grounding checked on {len(_CASES)} known cases")

_LANG_CASES = [
    ("how much urea should be applied in wheat", "en"),
    ("gehu me pila ratua lag gaya hai kya kare", "hinglish"),
    ("dhan ki nursery me pili patti ho rahi hai kya karein", "hinglish"),
    ("\u0917\u0947\u0939\u0942\u0902 \u092e\u0947\u0902 \u092a\u0940\u0932\u093e \u0930\u0924\u0941\u0906", "hi"),
    ("pm kisan samman nidhi eligibility and benefits", "en"),
    ("fall army worm control in maize", "en"),
]
for _q, _want in _LANG_CASES:
    _got = detect_lang(_q)
    assert _got == _want, f"detect_lang({_q!r}) gave {_got!r}, wanted {_want!r}"
print(f"detect_lang checked on {len(_LANG_CASES)} known cases "
      "(English, Hinglish and Hindi all separated)")

# Real answers from the first run, used as known cases for the two new checks.
_META_GOOD = "Spray mancozeb 2 g per litre of water.\nSources: [1], [3]"
_META_BAD  = ("\u092a\u0948\u0921\u0940 \u092b\u0938\u0932 \u092e\u0947\u0902 "
              "\u092c\u0915\u093e\u0928\u093f \u0930\u094b\u0917 \u0915\u0940 "
              "\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u0909\u092a\u0932\u092c\u094d\u0927 "
              "\u0939\u0948\u0964\nSources: [1], [2]")
_REFUSAL   = ("\u091c\u093e\u0928\u0915\u093e\u0930\u0940 \u0909\u092a\u0932\u092c\u094d\u0927 "
              "\u0928\u0939\u0940\u0902 \u0939\u0948\u0964\nSources: [1]")
assert not looks_like_meta_answer(_META_GOOD), "clean advice flagged as a meta answer"
assert looks_like_meta_answer(_META_BAD), "meta answer not caught"
assert not looks_like_meta_answer(_REFUSAL), "a proper refusal must not count as meta"
assert looks_like_refusal(_REFUSAL), "refusal not recognised"

assert not has_inline_citation("Spray mancozeb 2 g per litre.\nSources: [1], [3]")
assert has_inline_citation("Use 30 kg urea [1] and then irrigate.\nSources: [1]")
assert has_inline_citation("Information for Budaun [1] and Aligarh [2] is available.")

# Real answers from the smoke run. The half-refusals are the ones the earlier filter let
# through, so they are the important cases here.
_CLEAN_ADVICE = ("Spray Kartap Hydrochloride 50 SP, 400 grams mixed with 200 litres of "
                 "water per acre.\n\nSources: [4], [5]")
_CLEAN_REFUSE = ("I do not have information on how to treat rice seeds.\n\nSources: [1]")
_HALF_REFUSE  = ("I do not have information about plant protection for the Brinjal crop. "
                 "I can provide information about cultivation and planting times.\n\n"
                 "Sources: [1], [2]")
_TOPIC_LIST   = "Ask about fenugreek. Ask about strawberry. Ask about sugarcane.\n\nSources: [1]"
_CONSULT      = ("For sesame sowing, consult information about seed rate. You can also find "
                 "information about seed treatment.\n\nSources: [1]")
assert not looks_like_meta_answer(_CLEAN_ADVICE), "clean advice must be kept"
assert not looks_like_meta_answer(_CLEAN_REFUSE), "a plain refusal must be kept"
assert looks_like_refusal(_CLEAN_REFUSE)
assert looks_like_meta_answer(_HALF_REFUSE), "refuse-then-offer must be dropped"
assert looks_like_meta_answer(_TOPIC_LIST), "topic list must be dropped"
assert looks_like_meta_answer(_CONSULT), "consult-information must be dropped"
_MIXED = ("फसल सुरक्षा के "
          "लिए तारबंदी करें। "
          "यदि लहसुन की फसल "
          "में समस्या है। "
          "यदि बाजरा की फसल "
          "है। सरसों की फसल "
          "की जांच करें।")
assert mixes_crops(_MIXED), "the garlic/bajra/mustard answer must be caught"
assert not mixes_crops(_CLEAN_ADVICE), "single-crop advice must not be flagged"
print("meta / refusal / offer / crop-mixing checks pass on real smoke-run answers")

# Fixed replies the serving code sends without asking the model.
KVK_DISCLAIMER = {
    "en": "Please verify with your local KVK before applying.",
    "hinglish": "Apne nazdiki KVK se confirm karke hi use karein.",
    "hi": "\u0915\u0943\u092a\u092f\u093e \u0909\u092a\u092f\u094b\u0917 \u0938\u0947 "
          "\u092a\u0939\u0932\u0947 \u0905\u092a\u0928\u0947 \u0928\u091c\u0926\u0940\u0915\u0940 "
          "\u0915\u0947\u0935\u0940\u0915\u0947 \u0938\u0947 \u092a\u0941\u0937\u094d\u091f\u093f "
          "\u0915\u0930 \u0932\u0947\u0902\u0964",
}
ABSTAIN_MSG = {
    "en": ("This question is outside the agricultural knowledge base I can answer from. "
           "Please ask about crops, pests, fertilisers, or government agriculture schemes "
           "in Uttar Pradesh."),
    "hinglish": ("Yeh sawal meri kheti ki jankari se bahar hai. Kripya fasal, keet, khad ya "
                 "sarkari krishi yojana ke bare mein puchhein."),
    "hi": ("\u092f\u0939 \u092a\u094d\u0930\u0936\u094d\u0928 \u092e\u0947\u0930\u0947 "
           "\u0915\u0943\u0937\u093f \u091c\u094d\u091e\u093e\u0928 \u0906\u0927\u093e\u0930 "
           "\u0938\u0947 \u092c\u093e\u0939\u0930 \u0939\u0948\u0964 \u0915\u0943\u092a\u092f\u093e "
           "\u092b\u0938\u0932, \u0915\u0940\u091f, \u0909\u0930\u094d\u0935\u0930\u0915 "
           "\u092f\u093e \u0938\u0930\u0915\u093e\u0930\u0940 \u0915\u0943\u0937\u093f "
           "\u092f\u094b\u091c\u0928\u093e\u0913\u0902 \u0915\u0947 \u092c\u093e\u0930\u0947 "
           "\u092e\u0947\u0902 \u092a\u0942\u091b\u0947\u0902\u0964"),
}
print("prompt builder and scorers ready")


--- step 6 done in 0.0 min (total so far 2.4 min) ---

STEP 7/14  Set up prompts and scorers
expected ~0.7 min   |   about 59 min left of ~71 min total


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

prompt tokenizer: google/gemma-3-4b-it
numeric_grounding checked on 8 known cases
detect_lang checked on 6 known cases (English, Hinglish and Hindi all separated)
meta / refusal / offer / crop-mixing checks pass on real smoke-run answers
prompt builder and scorers ready


## 8. Load the KCC questions

Notebook 12 already filtered the full corpus and saved the good rows. We only need the
**questions** now — the KCC answers are kept for review, not for training.

Note the script-agreement filter in notebook 12 was written for the old design, where the KCC
answer was the training target and had to match the question's language. Now that the teacher
writes every answer, that filter only costs us questions. It is left alone because notebook 12
has already run; the effect is that our questions come from a narrower slice than they need to.
Worth removing in a later pass.

In [11]:
step(8, "Load the KCC questions")

SCHEME_HINTS = ("scheme", "subsidy", "insurance", "loan", "credit", "kisan credit")

def map_intent(query_type):
    """KCC's own scheme category was removed earlier, so field_practice is the right
    default and only the odd scheme-shaped row is switched."""
    qt = str(query_type or "").lower()
    return "policy" if any(h in qt for h in SCHEME_HINTS) else "field_practice"


if POOL_PATH:
    POOL = pd.read_csv(POOL_PATH)
    KCC_SOURCE = POOL_PATH
    print(f"loaded {len(POOL):,} questions from notebook 12")
else:
    print(f"[fallback] no kcc_pool.csv, filtering {KCC_RAW_PATH} here")
    _raw = pd.read_csv(KCC_RAW_PATH, low_memory=False)
    KCC_SOURCE = KCC_RAW_PATH
    _d = _raw.dropna(subset=["cleaned_query", "cleaned_answer"]).copy()
    _a = _d["cleaned_answer"].astype(str)
    _m = _a.str.len().between(40, 1200)
    _m &= ~_a.str.contains(r"\d{5,}", regex=True, na=False)
    _m &= ~_a.str.contains(r"call.{0,15}office|helpline", regex=True, case=False, na=False)
    _d = _d[_m]
    _d = _d[~_d["cleaned_query"].astype(str).str[:120].duplicated(keep="first")]
    POOL = _d.sample(n=min(len(_d), N_QUESTIONS * 4), random_state=SEED)
    print(f"built {len(POOL):,} questions here")

if len(POOL) < N_QUESTIONS:
    raise RuntimeError(
        f"only {len(POOL):,} questions available but {N_QUESTIONS:,} are needed.\n"
        f"Raise POOL_TARGET in notebook 12 and run it again, or lower N_QUESTIONS above.")

POOL = POOL.sample(frac=1, random_state=SEED).reset_index(drop=True)
POOL["intent"] = POOL["QueryType"].map(map_intent)

# itertuples() renames any column starting with "_" to a positional name, so a column called
# "_intent" would be unreachable as rec._intent. Keep names plain.
_bad = [c for c in POOL.columns if c.startswith("_")]
assert not _bad, f"columns {_bad} start with '_' and will break itertuples()"

print(f"\nintent mix: {dict(POOL['intent'].value_counts())}")
_langs = Counter(detect_lang(str(q)) for q in POOL["cleaned_query"].head(2000))
print(f"question languages (first 2000): {dict(_langs)}")
print("\nAs expected, KCC questions are almost all English. Step 9 fixes that.")


--- step 7 done in 0.2 min (total so far 2.6 min) ---

STEP 8/14  Load the KCC questions
expected ~0.8 min   |   about 58 min left of ~71 min total
loaded 6,000 questions from notebook 12

intent mix: {'field_practice': np.int64(6000)}
question languages (first 2000): {'en': 1846, 'hinglish': 154}

As expected, KCC questions are almost all English. Step 9 fixes that.


## 9. Rewrite some questions into Hindi and Hinglish

KCC questions are 99.98% English, so without this the student never sees a Hindi or Hinglish
question during training — while three quarters of our test set is non-English.

We load the small 4B model, ask it to rewrite a share of the questions, then unload it. Using
the small model keeps this cheap, and it runs **before** the teacher so only one large model is
in memory at a time.

The rewritten questions then go through normal search, so their context is retrieved from the
Hindi/Hinglish text itself — not borrowed from the English original.

A few rewrites are printed so you can see whether they look right. Set
`MAKE_OTHER_LANGUAGES = False` to skip this step.

In [12]:
step(9, "Rewrite questions into hi/hing")

# ---- Set up AIPIPE token (fail fast if missing) ----
import os
import time
import requests
from tqdm.auto import tqdm

if "AIPIPE_TOKEN" not in os.environ:
    # Try to load from Kaggle secrets
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["AIPIPE_TOKEN"] = UserSecretsClient().get_secret("AIPIPE_TOKEN")
    except ImportError:
        pass
if "AIPIPE_TOKEN" not in os.environ:
    raise RuntimeError(
        "AIPIPE_TOKEN not found. Please set it as an environment variable "
        "or add it to Kaggle Secrets."
    )

# ---- Select questions from the pool ----
QUESTIONS = []
for i, rec in enumerate(POOL.head(N_QUESTIONS).itertuples(index=False)):
    QUESTIONS.append({
        "qid": f"q{i:05d}",
        "query": str(rec.cleaned_query),
        "orig_query": str(rec.cleaned_query),
        "lang": "en",
        "intent": rec.intent,
        "kcc_reference_answer": str(rec.cleaned_answer),
        "meta": {"crop": rec.Crop, "district": rec.DistrictName,
                 "query_type": rec.QueryType, "year": rec.year},
    })
print(f"{len(QUESTIONS)} questions selected")

# ---- Helper: translate via aipipe ----
def translate_via_aipipe(text, target_lang, token=os.environ["AIPIPE_TOKEN"]):
    """Translate text to Hindi or Hinglish using Gemini via aipipe."""
    if target_lang == "hi":
        instruction = (
            "Translate this farmer's question into natural Hindi using Devanagari script. "
            "Keep the crop names in English (e.g., wheat, paddy, sugarcane, maize, potato, mustard). "
            "Reply with the translation only, nothing else."
        )
    else:  # hinglish
        instruction = (
            "Rewrite this farmer's question the way an Indian farmer would type it on a phone: "
            "Hindi words spelled in English letters (Hinglish). "
            "Keep the crop names in English. "
            "Reply with the rewritten question only, nothing else."
        )
    prompt = f"{instruction}\n\nQuestion: {text}"
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.2, "maxOutputTokens": 100}
    }
    headers = {
        "x-goog-api-key": token,
        "Content-Type": "application/json"
    }
    # Rate limit: ~15 RPM, so sleep 4 seconds between calls (conservative)
    time.sleep(4)
    response = requests.post(
        "https://aipipe.org/geminiv1beta/models/gemini-2.5-flash-lite:generateContent",
        headers=headers,
        json=payload,
        timeout=120
    )
    response.raise_for_status()
    data = response.json()
    try:
        result = data["candidates"][0]["content"]["parts"][0]["text"].strip()
    except (KeyError, IndexError):
        raise RuntimeError(f"Unexpected response from Gemini: {data}")
    return result

# ---- Perform translation if enabled ----
if MAKE_OTHER_LANGUAGES and (HINDI_FRACTION + HINGLISH_FRACTION) > 0:
    n_hi = int(len(QUESTIONS) * HINDI_FRACTION)
    n_hing = int(len(QUESTIONS) * HINGLISH_FRACTION)
    targets = ([("hi", q) for q in QUESTIONS[:n_hi]] +
               [("hinglish", q) for q in QUESTIONS[n_hi:n_hi + n_hing]])
    print(f"rewriting {n_hi} into Hindi and {n_hing} into Hinglish "
          f"({len(QUESTIONS)-n_hi-n_hing} stay English)")

    n_kept, n_rejected = 0, 0
    # Translate one by one (with rate limit already inside the function)
    for lang, q in tqdm(targets, desc="Translating"):
        try:
            translation = translate_via_aipipe(q["orig_query"], lang)
        except Exception as e:
            print(f"Translation failed for '{q['orig_query'][:60]}': {e}")
            n_rejected += 1
            continue
        # Basic validation
        if not translation or len(translation) < 5:
            n_rejected += 1
            continue
        # Check language (soft validation)
        detected = detect_lang(translation)
        if lang == "hi" and detected != "hi":
            n_rejected += 1
            continue
        if lang == "hinglish" and detected == "hi":
            n_rejected += 1
            continue
        # Accept
        q["query"] = translation
        q["lang"] = lang
        n_kept += 1
    print(f"kept {n_kept}, rejected {n_rejected} (wrong language or empty)")

    # Show a few samples
    print("\nsamples:")
    shown = 0
    for q in QUESTIONS:
        if q["lang"] != "en":
            print(f"   [{q['lang']:<8}] {q['orig_query'][:60]}")
            print(f"              -> {q['query'][:60]}")
            shown += 1
            if shown >= 4:
                break
else:
    print("MAKE_OTHER_LANGUAGES is off -- all questions stay English")

# ---- Final language mix ----
print(f"\nfinal question languages: {dict(Counter(q['lang'] for q in QUESTIONS))}")


--- step 8 done in 0.0 min (total so far 2.6 min) ---

STEP 9/14  Rewrite questions into hi/hing
expected ~0.1 min   |   about 57 min left of ~71 min total
2500 questions selected
rewriting 500 into Hindi and 500 into Hinglish (1500 stay English)


Translating:   0%|          | 0/1000 [00:00<?, ?it/s]

kept 996, rejected 4 (wrong language or empty)

samples:
   [hi      ] termite problem in cauliflower crop..
              -> फूलगोभी की फसल में दीमक की समस्या है..
   [hi      ] ASKED ABOUT INSECT CONTROL IN MINT ?
              -> पुदीना (mint) में कीड़े के नियंत्रण के बारे में पूछा गया है?
   [hi      ] Asked about plant protection
              -> पौध संरक्षण के बारे में पूछा गया
   [hi      ] Ask about sucking pests problem in crop cumin
              -> जी, क्या आपके cumin की फसल में चूसने वाले कीड़ों (sucking pe

final question languages: {'hi': 496, 'en': 1504, 'hinglish': 500}


## 10. Find context for every training question

One search per question. The prompt is built now, while the index is up, because notebook 14
has no index and cannot build it later.

Questions are dropped when the best match scores below the refuse threshold — there is nothing
to ground an answer in — or when the only match is the question's own KCC record, which would
teach the model to copy rather than to write.

In [13]:
step(10, "Search context for training Qs")
from tqdm.auto import tqdm

print(f"encoding {len(QUESTIONS)} questions ...")
_t0 = time.time()
_vecs = encode_queries([q["query"] for q in QUESTIONS])
print(f"encoded in {(time.time()-_t0)/60:.1f} min")

PROMPTS, drops = [], Counter()
_t0 = time.time()
for q, qv in tqdm(list(zip(QUESTIONS, _vecs)), desc="searching"):
    r = _search_core(q["query"], top_k=RETRIEVAL_TOP_K, intent=q["intent"], qvec=qv)
    if r["tier"] in ("error", "abstain_out_of_scope"):
        drops[r["tier"]] += 1
        continue
    own = f"Question: {q['orig_query']}\nAnswer: {q['kcc_reference_answer']}"
    hits = [h for h in r["results"] if h["text"].strip() != own.strip()]
    if not hits:
        drops["only_own_record"] += 1
        continue
    prompt, used = build_prompt(q["query"], hits, r["tier"], ctx_top_k=CTX_TOP_K,
                                char_cap=CTX_CHAR_CAP, lang=q["lang"])
    PROMPTS.append({**q, "prompt": prompt, "used": used,
                    "tier": r["tier"], "top_score": r["top_score"]})

print(f"\n{len(PROMPTS)} questions have context ({(time.time()-_t0)/60:.1f} min)")
print(f"dropped: {dict(drops)}")
print(f"by language: {dict(Counter(p['lang'] for p in PROMPTS))}")
print(f"by tier    : {dict(Counter(p['tier'] for p in PROMPTS))}")

if not PROMPTS:
    raise RuntimeError("no question kept any context. Check the search test in step 6.")

for p in PROMPTS:
    p["kind"] = "answer"

# ---- rows that teach the model to refuse ----------------------------------------------
OFF_TOPIC_QUESTIONS = [
    "how do I repair my motorcycle engine", "best chess opening for beginners",
    "who won the cricket world cup in 2011", "how to invest in the stock market",
    "what is the capital of France", "write a python function to sort a list",
    "how do I cook biryani", "what is the weather in Mumbai tomorrow",
    "explain quantum computing simply", "how to lose weight fast",
    "best mobile phone under 20000", "how do I apply for a passport",
    "who is the prime minister of India", "translate hello into French",
    "how to fix a leaking tap", "what is bitcoin",
    "\u092e\u094b\u092c\u093e\u0907\u0932 \u092b\u094b\u0928 \u0915\u0948\u0938\u0947 \u0920\u0940\u0915 \u0915\u0930\u0947\u0902",
    "\u0915\u094d\u0930\u093f\u0915\u0947\u091f \u092e\u0948\u091a \u0915\u093e \u0938\u094d\u0915\u094b\u0930 \u0915\u094d\u092f\u093e \u0939\u0948",
    "\u0936\u0947\u092f\u0930 \u092c\u093e\u091c\u093e\u0930 \u092e\u0947\u0902 \u092a\u0948\u0938\u093e \u0915\u0948\u0938\u0947 \u0932\u0917\u093e\u090f\u0902",
    "mera phone kharab ho gaya kya karu", "ghar ka kiraya kitna hona chahiye",
    "bank loan ke liye kya documents chahiye",
    "how to lose weight fast",
    "best mobile phone under 20000",
    "how do I apply for a passport",
    "who is the prime minister of India",
    "translate hello into French",
    "how to fix a leaking tap",
    "what is bitcoin",
    "how to start a YouTube channel",
    "how to learn English speaking",
    "best free antivirus for Windows",
    "how to change a car tyre",
    "how to make a website",
    "how to get a driving licence",
    "what is the highest mountain in the world",
    "how to play guitar",
    "how to meditate daily",
    "how to choose a mutual fund",
    "what is the best diet for diabetes",
    "how to book a train ticket online",
    "how to use Google Maps offline",
    "how to make a budget plan",
    "how to write a resume",
    "what is blockchain technology",
    "how to clean a laptop screen",
    "how to bake a cake",
    "how to study effectively for exams",
    "how to get better sleep",
    "what is the fastest animal on earth",
    "how to tie a tie",
    "how to speak in public without fear",
    "how to build a wooden shelf",
    "how to paint a room",
    "how to fix a bike puncture",
    "how to use a washing machine",
    "how to make coffee at home",
    "how to choose a life insurance policy",
    "how to file income tax return online",
    "how to open a bank account",
    "what is GST number",
    "how to pay electricity bill online",
    "how to book a flight ticket",
    "how to learn a new language fast",
    "how to become a pilot",
    "how to become a doctor",
    "how to start a small business",
    "how to get a home loan",
    "how to use Photoshop",
    "how to edit a video",
    "how to make a PowerPoint presentation",
    "how to write a cover letter",
    "how to negotiate salary",
    "how to manage stress",
    "how to stop smoking",
    "how to do yoga at home",
    "how to build muscle fast",
    "how to run a marathon",
    "how to swim for beginners",
    "how to ride a bicycle",
    "how to drive a car",
    "how to park a car",
    "how to parallel park",
    "how to change engine oil",
    "how to jump start a car",
    "how to use a fire extinguisher",
    "how to perform CPR",
    "how to treat a burn",
    "how to stop bleeding",
    "how to splint a broken bone",
    "how to survive in the wilderness",
    "how to build a campfire",
    "how to fish",
    "how to hunt",
    "how to ski",
    "how to surf",
    "how to scuba dive",
    "how to rock climb",
    "how to play football",
    "how to play basketball",
    "how to play tennis",
    "how to play cricket",
    "how to play badminton",
    "how to play table tennis",
    "how to play chess",
    "how to play poker",
    "how to play guitar chords",
    "how to play piano",
    "how to play drums",
    "how to sing better",
    "how to dance",
    "how to draw",
    "how to paint",
    "how to sculpt",
    "how to knit",
    "how to sew",
    "how to crochet",
    "how to make pottery",
    "how to do calligraphy",
    "how to take good photos",
    "how to make a video call",
    "how to use Zoom",
    "how to use Microsoft Teams",
    "how to use Slack",
    "how to use Trello",
    "how to use Excel formulas",
    "how to use Pivot Tables",
    "how to use VLOOKUP",
    "how to create a dashboard",
    "how to use Power BI",
    "how to use Tableau",
    "how to code in Python",
    "how to code in JavaScript",
    "how to code in Java",
    "how to code in C++",
    "how to build a website with HTML",
    "how to use CSS",
    "how to use React",
    "how to use Angular",
    "how to use Node.js",
    "how to use Django",
    "how to use Flask",
    "how to use Git",
    "how to use GitHub",
    "how to deploy a web app",
    "how to use AWS",
    "how to use Google Cloud",
    "how to use Azure",
    "how to set up a VPN",
    "how to secure your wifi",
    "how to prevent identity theft",
    "how to create a strong password",
    "how to backup your data",
    "how to recover deleted files",
    "how to format a hard drive",
    "how to install Windows",
    "how to install Ubuntu",
    "how to use a Mac",
    "how to use an iPhone",
    "how to use Android",
    "how to take a screenshot",
    "how to screen record",
    "how to transfer files from phone to computer",
    "how to print a document",
    "how to scan a document",
    "how to fax a document",
    "how to send an email",
    "how to write a formal letter",
    "how to write a complaint",
    "how to write a resignation letter",
    "how to write a thank you note",
    "how to apologise professionally",
    "how to give a speech",
    "how to do a job interview",
    "how to dress for an interview",
    "how to negotiate a job offer",
    "how to ask for a raise",
    "how to quit a job",
    "how to start a side hustle",
    "how to manage your time",
    "how to prioritise tasks",
    "how to set goals",
    "how to stay motivated",
    "how to overcome procrastination",
    "how to build self-discipline",
    "how to improve memory",
    "how to think critically",
    "how to solve a Rubik's cube",
    "how to do magic tricks",
    "how to do origami",
    "how to make a paper plane",
    "how to fold a fitted sheet",
    "how to remove stains from clothes",
    "how to clean silver jewellery",
    "how to sharpen a knife",
    "how to store vegetables",
    "how to meal prep",
    "how to make a smoothie",
    "how to cook rice perfectly",
    "how to make pasta from scratch",
    "how to bake bread",
    "how to make pizza",
    "how to brew beer",
    "how to make wine",
    "how to make cheese",
    "how to make chocolate",
    "how to grow a vegetable garden",
    "how to grow flowers",
    "how to prune trees",
    "how to cut grass",
    "how to repair a fence",
    "how to build a deck",
    "how to insulate a house",
    "how to fix a roof",
    "how to unclog a drain",
    "how to fix a toilet",
    "how to wire a plug",
    "how to replace a light fixture",
    "how to install a ceiling fan",
    "how to hang a picture",
    "how to wallpaper a room",
    "how to lay tiles",
    "how to install carpet",
    "how to paint a fence",
    "how to restore furniture",
    "how to make a birdhouse",
    "how to make a dog house",
    "how to make a kite",
    "how to make a boomerang",
    "how to make a bow and arrow",
    "how to make a catapult",
    "how to make a volcano for science fair",
    "how to make a solar oven",
    "how to make a water filter",
    "how to make a battery",
    "how to make a generator",
    "how to make a speaker",
    "how to make a robot",
    "how to make a drone",
    "how to make a 3D print model",
    "how to use a 3D printer",
    "how to use a laser cutter",
    "how to use a CNC machine",
    "how to weld metal",
    "how to solder electronics",
    "how to fix a smartphone screen",
    "how to replace a laptop battery",
    "how to upgrade RAM",
    "how to install an SSD",
    "how to build a gaming PC"
]

REFUSAL_PROMPTS = []
if ADD_REFUSAL_ROWS and PROMPTS:
    # (a) real question, wrong context. Pairing question i with question i+7's context gives
    #     a plausible-looking farming context that does not answer the question asked.
    n_swap = int(len(PROMPTS) * NO_CONTEXT_FRACTION)
    for i in range(n_swap):
        src_q = PROMPTS[i]
        other = PROMPTS[(i + 7) % len(PROMPTS)]
        if other["qid"] == src_q["qid"]:
            continue
        prompt, _ = build_prompt(src_q["query"], [{"text": h["text"], "raw_score": h["raw_score"],
                                                   "chunk_id": h["chunk_id"],
                                                   "source_type": h["source_type"],
                                                   "citation": h["citation"]}
                                                  for h in other["used"]],
                                 "fallback_with_disclaimer", ctx_top_k=CTX_TOP_K,
                                 char_cap=CTX_CHAR_CAP, lang=src_q["lang"])
        REFUSAL_PROMPTS.append({
            "qid": f"nc{i:05d}", "kind": "no_context", "query": src_q["query"],
            "orig_query": src_q["orig_query"], "lang": src_q["lang"],
            "intent": src_q["intent"], "tier": "fallback_with_disclaimer",
            "top_score": 0.0, "prompt": prompt, "used": other["used"],
            "kcc_reference_answer": "", "meta": src_q["meta"],
        })

    # (b) off-topic question, no context at all
    for i, q in enumerate(OFF_TOPIC_QUESTIONS[:OFF_TOPIC_COUNT]):
        lang = detect_lang(q)
        prompt, _ = build_prompt(q, [], "abstain_out_of_scope", ctx_top_k=CTX_TOP_K,
                                 char_cap=CTX_CHAR_CAP, lang=lang)
        REFUSAL_PROMPTS.append({
            "qid": f"ot{i:05d}", "kind": "off_topic", "query": q, "orig_query": q,
            "lang": lang, "intent": "general", "tier": "abstain_out_of_scope",
            "top_score": 0.0, "prompt": prompt, "used": [],
            "kcc_reference_answer": "", "meta": {},
        })

    PROMPTS.extend(REFUSAL_PROMPTS)
    print(f"\nadded {len(REFUSAL_PROMPTS)} refusal rows "
          f"({dict(Counter(r['kind'] for r in REFUSAL_PROMPTS))})")
    print(f"total prompts now {len(PROMPTS)}, "
          f"{len(REFUSAL_PROMPTS)/len(PROMPTS):.0%} of them teach refusal")
else:
    print("\nADD_REFUSAL_ROWS is off -- the model will never practise saying "
          "'I do not have that'")

_tok_lens = [len(gen_tok(p["prompt"], add_special_tokens=False)["input_ids"])
             for p in PROMPTS[:64]]
print(f"prompt length (first 64): median {sorted(_tok_lens)[len(_tok_lens)//2]} tokens, "
      f"max {max(_tok_lens)}")


--- step 9 done in 78.8 min (total so far 81.4 min) ---

STEP 10/14  Search context for training Qs
expected ~6.0 min   |   about 57 min left of ~71 min total
encoding 2500 questions ...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

encoded in 0.2 min


searching:   0%|          | 0/2500 [00:00<?, ?it/s]


2495 questions have context (0.6 min)
dropped: {'abstain_out_of_scope': 5}
by language: {'hi': 496, 'en': 1504, 'hinglish': 495}
by tier    : {'grounded': 2334, 'fallback_with_disclaimer': 161}

added 873 refusal rows ({'no_context': 623, 'off_topic': 250})
total prompts now 3368, 26% of them teach refusal
prompt length (first 64): median 1409 tokens, max 1746


## 11. Find context for the test questions

Built now for the same reason: notebook 14 has no index.

Two prompts are saved per question — the normal one, and one with **empty context**. The second
is a control. Without it, a model answering from memory looks the same as one answering from
evidence.

Off-domain questions should score below the refuse threshold and never reach the model at all.

In [14]:
step(11, "Search context for eval Qs")

EVAL_SET = [
    # policy
    dict(q="who is eligible for interest subvention on crop loans", group="policy", intent="policy"),
    dict(q="pm kisan samman nidhi eligibility and benefits", group="policy", intent="policy"),
    dict(q="what documents are needed for crop insurance claim", group="policy", intent="policy"),
    dict(q="soil health card scheme how to apply", group="policy", intent="policy"),
    # field practice, English
    dict(q="how much urea should be applied in wheat at tillering stage", group="field", intent="field_practice"),
    dict(q="fall army worm control in maize", group="field", intent="field_practice"),
    dict(q="red rot disease treatment in sugarcane", group="field", intent="field_practice"),
    dict(q="best fertilizer dose for paddy nursery", group="field", intent="field_practice"),
    dict(q="late blight control in potato crop", group="field", intent="field_practice"),
    dict(q="aphid attack on mustard what to spray", group="field", intent="field_practice"),
    # Hindi
    dict(q="\u0917\u0947\u0939\u0942\u0902 \u092e\u0947\u0902 \u092a\u0940\u0932\u093e \u0930\u0924\u0941\u0906 \u0915\u0940 \u0930\u094b\u0915\u0925\u093e\u092e \u0915\u0948\u0938\u0947 \u0915\u0930\u0947\u0902", group="hindi", intent="field_practice"),
    dict(q="\u0927\u093e\u0928 \u0915\u0940 \u0928\u0930\u094d\u0938\u0930\u0940 \u092e\u0947\u0902 \u092a\u0940\u0932\u0940 \u092a\u0924\u094d\u0924\u0940 \u0939\u094b \u0930\u0939\u0940 \u0939\u0948", group="hindi", intent="field_practice"),
    dict(q="\u091f\u092e\u093e\u091f\u0930 \u0915\u0947 \u092a\u094c\u0927\u0947 \u092e\u0947\u0902 \u092a\u0924\u094d\u0924\u093f\u092f\u093e\u0902 \u092e\u0941\u0921\u093c \u0930\u0939\u0940 \u0939\u0948\u0902 \u0915\u094d\u092f\u093e \u0915\u0930\u0947\u0902", group="hindi", intent="field_practice"),
    dict(q="\u0917\u0928\u094d\u0928\u0947 \u092e\u0947\u0902 \u0932\u093e\u0932 \u0938\u0921\u093c\u0928 \u0930\u094b\u0917 \u0915\u093e \u0907\u0932\u093e\u091c", group="hindi", intent="field_practice"),
    # Hinglish
    dict(q="dhan ki nursery me pili patti ho rahi hai kya karein", group="hinglish", intent="field_practice"),
    dict(q="gehu me pila ratua lag gaya hai kya kare", group="hinglish", intent="field_practice"),
    dict(q="aloo ki fasal me jhulsa rog ki dawa bataye", group="hinglish", intent="field_practice"),
    dict(q="ganne me kide lag gaye hain kaun si dawa dale", group="hinglish", intent="field_practice"),
    # known gaps -- no market price data exists in the index
    dict(q="mandi bhav for wheat today in lucknow", group="gap", intent="general"),
    dict(q="onion market price today uttar pradesh", group="gap", intent="general"),
    # off domain, must refuse
    dict(q="how do I repair my motorcycle engine", group="offdomain", intent="general"),
    dict(q="best chess opening strategy for beginners", group="offdomain", intent="general"),
    dict(q="\u0936\u0947\u092f\u0930 \u092c\u093e\u091c\u093e\u0930 \u092e\u0947\u0902 \u0928\u093f\u0935\u0947\u0936 \u0915\u0948\u0938\u0947 \u0915\u0930\u0947\u0902", group="offdomain", intent="general"),
    dict(q="who won the cricket world cup in 2011", group="offdomain", intent="general"),
    # messy and vague
    dict(q="sarkar ki sbse nayi scheme koinsi hai", group="messy", intent="policy"),
    dict(q="hi how are you", group="messy", intent="general"),
    dict(q="i need to know latest farming practices and which crops could i benefit from if am "
           "in UP and also i want to know about government policies to beneift from",
         group="messy", intent="general"),
    dict(q="kya karu fasal kharab ho gayi", group="messy", intent="field_practice"),
    # the two named checks
    dict(q="pm kisan samman nidhi eligibility and benefits", group="acceptance", intent="policy",
         note="must not state Rs 6000 unless a retrieved chunk contains it"),
    dict(q="PM Kisan ke liye age limit kya hai", group="acceptance", intent="policy",
         note="must not answer with PM-Kisan Maan-Dhan's 18-40 pension rule"),
]

_train_qs = {p["query"].strip().lower() for p in PROMPTS}
_before = len(EVAL_SET)
EVAL_SET = [e for e in EVAL_SET if e["q"].strip().lower() not in _train_qs]
if len(EVAL_SET) != _before:
    print(f"removed {_before-len(EVAL_SET)} test questions that also appear in training")

_e_vecs = encode_queries([e["q"] for e in EVAL_SET])
EVAL_PREPPED = []
for e, qv in zip(EVAL_SET, _e_vecs):
    r = _search_core(e["q"], top_k=CTX_TOP_K, intent=e.get("intent", "general"), qvec=qv)
    rec = {"q": e["q"], "group": e["group"], "intent": e.get("intent", "general"),
           "note": e.get("note"), "lang": detect_lang(e["q"]),
           "tier": r["tier"], "top_score": r["top_score"], "error": r.get("error")}
    if r["tier"] not in ("error", "abstain_out_of_scope"):
        prompt, used = build_prompt(e["q"], r["results"], r["tier"], ctx_top_k=CTX_TOP_K,
                                    char_cap=CTX_CHAR_CAP)
        nr_prompt, _ = build_prompt(e["q"], [], "abstain_out_of_scope", ctx_top_k=CTX_TOP_K)
        rec.update(prompt=prompt, no_retrieval_prompt=nr_prompt, used=used)
    EVAL_PREPPED.append(rec)

print(f"{len(EVAL_PREPPED)} test questions ready, "
      f"{sum(1 for r in EVAL_PREPPED if 'prompt' in r)} will reach the model")
print(f"by language: {dict(Counter(r['lang'] for r in EVAL_PREPPED))}")
print(f"by tier    : {dict(Counter(r['tier'] for r in EVAL_PREPPED))}")
_off = [r for r in EVAL_PREPPED if r["group"] == "offdomain"]
print(f"off-domain refused before reaching the model: "
      f"{sum(1 for r in _off if r['tier']=='abstain_out_of_scope')}/{len(_off)}")


--- step 10 done in 0.8 min (total so far 82.2 min) ---

STEP 11/14  Search context for eval Qs
expected ~0.3 min   |   about 51 min left of ~71 min total
removed 2 test questions that also appear in training


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

28 test questions ready, 24 will reach the model
by language: {'en': 15, 'hi': 5, 'hinglish': 8}
by tier    : {'grounded': 13, 'fallback_with_disclaimer': 11, 'abstain_out_of_scope': 4}
off-domain refused before reaching the model: 2/2


## 12. Shut down the search and save progress

Everything that needs the index is done. Shutting it down frees about 2.5 GB of GPU memory and
several GB of disk before the teacher loads.

We also save the work so far. Everything above is slow and needs the index; everything below is
the teacher, which is where a session is most likely to die. If that happens you can restart and
run **step 0, step 7 and step 13 only** — step 13 reloads this file.

In [15]:
step(12, "Shut down search, save progress")
vram_report("before shutdown")

free_vram("embed_model")
try:
    qdrant_client.close()
except Exception:
    pass
if qdrant_proc is not None:
    try:
        qdrant_proc.terminate()
        qdrant_proc.wait(timeout=60)
        print("qdrant stopped")
    except Exception as e:
        print(f"(qdrant was already stopped: {type(e).__name__})")

shutil.rmtree(QDRANT_STORAGE, ignore_errors=True)
shutil.rmtree(QDRANT_SNAPDIR, ignore_errors=True)
print("qdrant files removed")

CHECKPOINT = os.path.join(OUTPUT_DIR, "search_stage.json")
with open(CHECKPOINT, "w", encoding="utf-8") as f:
    json.dump({"prompts": PROMPTS, "eval_prepped": EVAL_PREPPED,
               "drops": dict(drops)}, f, ensure_ascii=False)
print(f"progress saved to {CHECKPOINT} ({os.path.getsize(CHECKPOINT)/1e6:.1f} MB)")

vram_report("after shutdown")


--- step 11 done in 0.0 min (total so far 82.2 min) ---

STEP 12/14  Shut down search, save progress
expected ~0.5 min   |   about 51 min left of ~71 min total
[before shutdown] gpu0 12.9 GB free   gpu1 15.5 GB free   disk 13.2 GB free
qdrant stopped
qdrant files removed
progress saved to /kaggle/working/distill_data/search_stage.json (27.5 MB)
[after shutdown] gpu0 15.5 GB free   gpu1 15.5 GB free   disk 20.8 GB free


## 13. The teacher writes the answers

This is the long step. Watch the progress bar for a time estimate.

Three things keep the session alive:

* **Disk check before the download**, so a full disk is found in seconds not minutes.
* **Memory check before loading**, because re-running a load cell adds a second copy of the
  model without freeing the first. Only restarting the session clears that.
* **Answers are written to disk after every batch.** If the session dies at question 400 of 800,
  re-running this cell starts again at 400.

Answers whose numbers do not all appear in the context are dropped. That catches invented
figures only — a wrong meaning passes. Every row keeps the KCC expert answer beside it so a
person can compare.

In [16]:
step(13, "Teacher writes the answers")

import os
import time
import json
import requests
from tqdm.auto import tqdm

# ---- Ensure AIPIPE_TOKEN is available ----
if "AIPIPE_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["AIPIPE_TOKEN"] = UserSecretsClient().get_secret("AIPIPE_TOKEN")
    except ImportError:
        pass
if "AIPIPE_TOKEN" not in os.environ:
    raise RuntimeError("AIPIPE_TOKEN not found. Please set it as an environment variable or Kaggle secret.")

# ---- Helper: call Gemini via aipipe ----
def teacher_generate(prompt, token=os.environ["AIPIPE_TOKEN"]):
    """Send a prompt to Gemini via aipipe and return the completion."""
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "temperature": TEACHER_TEMPERATURE,
            "maxOutputTokens": ANSWER_MAX_TOKENS,
            "top_p": 0.9
        }
    }
    headers = {
        "x-goog-api-key": token,
        "Content-Type": "application/json"
    }
    # Rate limit: sleep 4 seconds between calls (~15 RPM)
    time.sleep(4)
    response = requests.post(
        "https://aipipe.org/geminiv1beta/models/gemini-2.5-flash-lite:generateContent",
        headers=headers,
        json=payload,
        timeout=180
    )
    response.raise_for_status()
    data = response.json()
    try:
        text = data["candidates"][0]["content"]["parts"][0]["text"].strip()
    except (KeyError, IndexError):
        raise RuntimeError(f"Unexpected response from Gemini: {data}")
    return text

# ---- Resume from checkpoint if available ----
ROWS = []
TEACHER_QUANT = "aipipe-gemini-2.5-flash-lite"  # Record what we used
RAW_PATH = os.path.join(OUTPUT_DIR, "teacher_answers_raw.jsonl")

# If this is a fresh session after a crash, reload what step 12 saved.
if "PROMPTS" not in globals():
    _ck = os.path.join(OUTPUT_DIR, "search_stage.json")
    if not os.path.exists(_ck):
        raise RuntimeError(f"nothing in memory and no checkpoint at {_ck}. Run the notebook from step 1.")
    _d = json.load(open(_ck, encoding="utf-8"))
    PROMPTS, EVAL_PREPPED = _d["prompts"], _d["eval_prepped"]
    drops = Counter(_d["drops"])
    print(f"[resume] loaded {len(PROMPTS)} prompts from the checkpoint")

# Pick up any answers from a previous partial run.
answers = {}
if os.path.exists(RAW_PATH):
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                d = json.loads(line)
                answers[d["qid"]] = d["answer"]
    print(f"[resume] {len(answers)} answers already on disk")

todo = [p for p in PROMPTS if p["qid"] not in answers]
print(f"{len(todo)} answers still to write")

# ---- Teacher writes the answers ----
if todo:
    # Open the raw answers file in append mode
    with open(RAW_PATH, "a", encoding="utf-8") as f_out:
        for i, p in enumerate(tqdm(todo, desc="Teacher generating")):
            try:
                ans = teacher_generate(p["prompt"])
            except Exception as e:
                print(f"Failed for {p['qid']} ({p['query'][:60]}): {e}")
                continue
            # Store in memory
            answers[p["qid"]] = ans
            # Write immediately to disk (crash recovery)
            f_out.write(json.dumps({"qid": p["qid"], "answer": ans}, ensure_ascii=False) + "\n")
            f_out.flush()
            # Progress update every 50
            if (i + 1) % 50 == 0:
                elapsed = (time.time() - RUN_START) / 60
                print(f"  {i+1}/{len(todo)} done, {elapsed:.1f} min elapsed")

    print(f"writing complete. Total answers: {len(answers)}")

# ---- Keep the answers that pass the filters (same as before) ----
n_dropped, n_missing, n_no_refusal = 0, 0, 0
n_meta, n_inline, n_nocite, n_mixed = 0, 0, 0, 0
for p in PROMPTS:
    ans = answers.get(p["qid"])
    if not ans:
        n_missing += 1
        continue
    kind = p.get("kind", "answer")
    ctx_blob = " ".join(h["text"] for h in p["used"])

    if kind == "answer":
        # Keep only answers whose numbers all appear in the context.
        grounding, unsupported = numeric_grounding(ans, ctx_blob)
        if grounding < 1.0:
            n_dropped += 1
            continue
        # Drop answers that describe where information lives instead of advising.
        if looks_like_meta_answer(ans):
            n_meta += 1
            continue
        if mixes_crops(ans):
            n_mixed += 1
            continue
        # Drop answers with citations inside the sentences.
        if has_inline_citation(ans):
            n_inline += 1
            continue
        if REQUIRE_CITATION and not has_citation(ans):
            n_nocite += 1
            continue
    else:
        # Refusal rows: keep only if the teacher actually refused.
        if not looks_like_refusal(ans):
            n_no_refusal += 1
            continue
        # A refusal that then offers other topics is also bad.
        if offers_alternatives(ans):
            n_meta += 1
            continue
        grounding = 1.0

    ROWS.append({
        "id": p["qid"], "kind": kind, "query": p["query"], "orig_query": p["orig_query"],
        "lang": p["lang"], "intent": p["intent"], "tier": p["tier"],
        "top_score": p["top_score"], "prompt": p["prompt"], "completion": ans,
        "retrieved_chunk_ids": [h["chunk_id"] for h in p["used"]],
        "context_texts": [h["text"] for h in p["used"]],
        "teacher_grounding": grounding,
        "teacher_model": "gemini-2.5-flash-lite (via aipipe)",
        "kcc_reference_answer": p["kcc_reference_answer"],
        "meta": p["meta"],
    })

_started = len(PROMPTS)
print(f"\nkept {len(ROWS)} rows out of {_started} prompts")
print(f"   {'dropped, numbers not in context':<42} {n_dropped}")
print(f"   {'dropped, described where info lives':<42} {n_meta}")
print(f"   {'dropped, mixed several crops together':<42} {n_mixed}")
print(f"   {'dropped, citations inside sentences':<42} {n_inline}")
print(f"   {'dropped, no Sources line':<42} {n_nocite}")
print(f"   {'dropped, refusal row but teacher answered':<42} {n_no_refusal}")
print(f"   {'never written':<42} {n_missing}")
print(f"   {'KEPT':<42} {len(ROWS)}  ({len(ROWS)/max(_started,1):.0%})")
print(f"   kinds: {dict(Counter(r['kind'] for r in ROWS))}")

# ---- Quality report (language match, citations, etc.) ----
scored = [(p["query"], p["lang"], answers.get(p["qid"])) for p in PROMPTS if answers.get(p["qid"])]
if scored:
    print(f"\nteacher answer quality ({len(scored)} answers):")
    print(f"  {'language':<10} {'count':>6} {'right language':>16} {'has citation':>14}")
    for lang in ("en", "hinglish", "hi"):
        sub = [(q, a) for q, l, a in scored if l == lang]
        if not sub:
            continue
        lm = sum(1 for q, a in sub if language_match(q, a) == 1.0) / len(sub)
        ct = sum(1 for q, a in sub if has_citation(a)) / len(sub)
        print(f"  {lang:<10} {len(sub):>6} {lm:>15.0%} {ct:>13.0%}")
    lm_all = sum(1 for q, l, a in scored if language_match(q, a) == 1.0) / len(scored)
    ct_all = sum(1 for q, l, a in scored if has_citation(a)) / len(scored)
    print(f"  {'all':<10} {len(scored):>6} {lm_all:>15.0%} {ct_all:>13.0%}")

    _meta = sum(1 for q, l, a in scored if looks_like_meta_answer(a)) / len(scored)
    _inl  = sum(1 for q, l, a in scored if has_inline_citation(a)) / len(scored)
    print(f"\n  described where info lives : {_meta:.0%}  (was very common before the SOURCE/CONTENT split)")
    print(f"  citations inside sentences: {_inl:.0%}  (rule 5 says last line only)")

    hi_rows = [(q, a) for q, l, a in scored if l == "hi"]
    if hi_rows and sum(1 for q, a in hi_rows if language_match(q, a) == 1.0) / len(hi_rows) < 0.7:
        print("\n  [WARN] the teacher answers Hindi questions in the wrong script more than 30% of the time.")
    if ct_all < 0.7:
        print("\n  [WARN] fewer than 70% of answers carry a citation. Try raising ANSWER_MAX_TOKENS.")


--- step 12 done in 0.0 min (total so far 82.3 min) ---

STEP 13/14  Teacher writes the answers
expected ~50.0 min   |   about 50 min left of ~71 min total
3368 answers still to write


Teacher generating:   0%|          | 0/3368 [00:00<?, ?it/s]

  50/3368 done, 86.4 min elapsed
  100/3368 done, 90.4 min elapsed
  150/3368 done, 94.9 min elapsed
  200/3368 done, 99.0 min elapsed
  250/3368 done, 103.1 min elapsed
  300/3368 done, 107.2 min elapsed
  350/3368 done, 111.3 min elapsed
  400/3368 done, 115.5 min elapsed
  450/3368 done, 119.6 min elapsed
  500/3368 done, 123.7 min elapsed
  550/3368 done, 127.8 min elapsed
  600/3368 done, 131.8 min elapsed
  650/3368 done, 135.9 min elapsed
  700/3368 done, 140.0 min elapsed
  750/3368 done, 144.0 min elapsed
  800/3368 done, 148.1 min elapsed
  850/3368 done, 152.2 min elapsed
  900/3368 done, 156.2 min elapsed
  950/3368 done, 160.3 min elapsed
  1000/3368 done, 164.4 min elapsed
  1050/3368 done, 168.4 min elapsed
  1100/3368 done, 172.4 min elapsed
  1150/3368 done, 176.5 min elapsed
  1200/3368 done, 180.4 min elapsed
  1250/3368 done, 184.5 min elapsed
  1300/3368 done, 188.5 min elapsed
  1350/3368 done, 192.5 min elapsed
  1400/3368 done, 196.5 min elapsed
  1450/3368 done

## 14. Save the files

Four files leave this notebook. `train.jsonl` is the training data. `eval_prepped.jsonl` is the
one that cannot be rebuilt later, because it holds prompts that needed the index. `review.csv`
puts the teacher's answer next to the KCC expert answer so a person can compare them.

The config hash links everything: notebook 14 records it, so any number can be traced back to
the data it came from.

In [17]:
step(14, "Save the output files")

random.Random(SEED).shuffle(ROWS)

train_path = os.path.join(OUTPUT_DIR, "train.jsonl")
with open(train_path, "w", encoding="utf-8") as f:
    for r in ROWS:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

eval_path = os.path.join(OUTPUT_DIR, "eval_prepped.jsonl")
with open(eval_path, "w", encoding="utf-8") as f:
    for r in EVAL_PREPPED:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

if ROWS:
    pd.DataFrame([{"query": r["query"], "kind": r["kind"], "lang": r["lang"],
                   "intent": r["intent"],
                   "tier": r["tier"], "top_score": r["top_score"],
                   "teacher_answer": r["completion"],
                   "kcc_expert_answer": r["kcc_reference_answer"],
                   "crop": r["meta"].get("crop")} for r in ROWS]
                 ).to_csv(os.path.join(OUTPUT_DIR, "review.csv"), index=False)

config_str = json.dumps(dict(
    n_rows=len(ROWS), student_tokenizer=STUDENT_MODEL_ID, teacher=TEACHER_MODEL_ID,
    answer_max_tokens=ANSWER_MAX_TOKENS, teacher_temperature=TEACHER_TEMPERATURE,
    ctx_top_k=CTX_TOP_K, ctx_char_cap=CTX_CHAR_CAP, seed=SEED,
    hindi_fraction=HINDI_FRACTION, hinglish_fraction=HINGLISH_FRACTION,
    embed_model=MANIFEST["embed_model"], tiers=MANIFEST["tiers"],
), sort_keys=True)
DATASET_CONFIG_HASH = hashlib.sha256(config_str.encode()).hexdigest()[:12]

finish_steps()
manifest = {
    "config_hash": DATASET_CONFIG_HASH,
    "built_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "stage": "13_distill_dataset_kaggle",
    "smoke_test": SMOKE_TEST,
    "method": "sequence-level knowledge distillation. Every training answer is written by the "
              "teacher over retrieved context. The KCC expert answer is stored for review "
              "only and is never a training target, because those answers were written "
              "without any retrieved context and would teach the model to skip citations "
              "and state unsupported numbers.",
    "n_rows": len(ROWS),
    "n_questions_started": len(QUESTIONS) if "QUESTIONS" in globals() else None,
    "n_dropped_unsupported_numbers": n_dropped,
    "n_dropped_teacher_did_not_refuse": n_no_refusal,
    "n_dropped_meta_answer_or_offered_alternatives": n_meta,
    "n_dropped_inline_citation": n_inline,
    "n_dropped_mixed_crops": n_mixed,
    "n_dropped_no_citation": n_nocite,
    "row_kinds": dict(Counter(r["kind"] for r in ROWS)),
    "refusal_rows_added": ADD_REFUSAL_ROWS,
    "language_counts": dict(Counter(r["lang"] for r in ROWS)),
    "tier_counts": dict(Counter(r["tier"] for r in ROWS)),
    "teacher_model_id": TEACHER_MODEL_ID,
    "teacher_quantization": TEACHER_QUANT,
    "student_tokenizer_id": STUDENT_MODEL_ID,
    "answer_max_tokens": ANSWER_MAX_TOKENS,
    "ctx_top_k": CTX_TOP_K, "ctx_char_cap": CTX_CHAR_CAP,
    "made_other_languages": MAKE_OTHER_LANGUAGES,
    "hindi_fraction": HINDI_FRACTION, "hinglish_fraction": HINGLISH_FRACTION,
    "kcc_source": os.path.basename(KCC_SOURCE),
    "pdf_used_as_training_source": False,
    "index": {"embed_model": MANIFEST["embed_model"], "embed_dim": MANIFEST["embed_dim"],
              "collection": MANIFEST["collection"], "n_chunks": MANIFEST.get("n_chunks"),
              "tiers": MANIFEST["tiers"]},
    "search_drops": dict(drops),
    "eval_rows": len(EVAL_PREPPED),
    "eval_groups": dict(Counter(r["group"] for r in EVAL_PREPPED)),
    "step_minutes": {f"{n}. {name}": mins for n, name, mins in STEP_LOG},
    "total_minutes": round((time.time() - RUN_START) / 60, 1),
    "seed": SEED,
}
with open(os.path.join(OUTPUT_DIR, "dataset_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=" * 78)
print("DONE")
print("=" * 78)
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {fn:<30} {os.path.getsize(os.path.join(OUTPUT_DIR, fn))/1e6:>8,.2f} MB")

print(f"\nconfig hash   : {DATASET_CONFIG_HASH}")
print(f"training rows : {len(ROWS)}")
print(f"languages     : {dict(Counter(r['lang'] for r in ROWS))}")
print(f"test rows     : {len(EVAL_PREPPED)}")

print("\ntime per step:")
for n, name, mins in STEP_LOG:
    print(f"   {n:>2}. {name:<34} {mins:>6.1f} min")
print(f"   {'':>2}  {'TOTAL':<34} {(time.time()-RUN_START)/60:>6.1f} min")

if ROWS:
    _tk = sorted(len(gen_tok(r["prompt"] + " " + r["completion"],
                             add_special_tokens=False)["input_ids"])
                 for r in ROWS[:200])
    print(f"\nprompt+answer length: median {_tk[len(_tk)//2]}, "
          f"95th percentile {_tk[int(len(_tk)*0.95)]}, max {_tk[-1]} tokens")
    print("   -> MAX_SEQ_LEN in notebook 14 should be at least the 95th percentile.")

print("""
NEXT
----
1. Save Version -> "Save & Run All (Commit)".
2. Open 14_distill_train_eval_kaggle -> Add Input -> Your Work -> this notebook's output.
3. Settings: GPU T4 x2, Internet On, HF_TOKEN attached.
""")
if SMOKE_TEST:
    print("!! This was a smoke test. Set SMOKE_TEST = False and run again for real data.")


--- step 13 done in 271.7 min (total so far 353.9 min) ---

STEP 14/14  Save the output files
expected ~0.5 min   |   about 0 min left of ~71 min total
DONE
  dataset_manifest.json              0.00 MB
  eval_prepped.jsonl                 0.34 MB
  review.csv                         1.25 MB
  search_stage.json                 27.51 MB
  teacher_answers_raw.jsonl          0.75 MB
  train.jsonl                       23.15 MB

config hash   : 1d8f4ed7d4ca
training rows : 3187
languages     : {'en': 1708, 'hinglish': 558, 'hi': 921}
test rows     : 28

time per step:
    1. Find the input files                  0.0 min
    2. Start the Qdrant server               0.0 min
    3. Load the search index                 1.3 min
    4. Load the embedding model              1.0 min
    5. Set up name cleanup                   0.0 min
    6. Set up the search function            0.0 min
    7. Set up prompts and scorers            0.2 min
    8. Load the KCC questions                0.0 min
    9

In [18]:
print(f"AIPIPE_TOKEN present: {bool(os.environ.get('AIPIPE_TOKEN'))}")
print(f"Token starts with: {os.environ.get('AIPIPE_TOKEN', '')[:10]}...")

AIPIPE_TOKEN present: True
Token starts with: eyJhbGciOi...
